# Wildlife Re-Identification · Leopard spatial blend

**Kaggle public MAP@5** was **0.84751** for this submission.

This notebook reproduces `submission_portfolio_leopard_v3.csv`. Each of the 2,791
test queries receives five distinct, ranked identities, including `new_individual`
when applicable. The model uses the supplied descriptors, SIFT keypoints, and metadata.

## Method

The species-specific portfolio combines LightGBM, histogram gradient boosting,
CatBoost and XGBoost with camera-side calibration and SIFT-verified graph pooling.
All hyena rankings come from that portfolio. For leopards, the candidate combines 55%
of the portfolio's conditional known-identity distribution and 45% of the spatial
model's conditional known-identity distribution. The spatial model uses spatial
Fisher/VLAD, Gaussian PLDA, and MAGSAC-verified SIFT matches. Its weight is quartered
for gallery identities with one or two photos. The portfolio supplies the full
novelty probability for both species.

## Validation and error analysis

| Validation split | Portfolio MAP@5 | Leopard blend MAP@5 | Rare known, portfolio | Rare known, blend |
|---|---:|---:|---:|---:|
| Shared | 0.89384 | 0.89415 | 0.45806 | 0.45806 |
| Encounter clusters | 0.89768 | 0.89969 | 0.45359 | 0.45359 |
| Ordinary audit | 0.89605 | 0.89642 | 0.36984 | 0.36984 |
| Withheld viewpoint | 0.80631 | 0.80741 | 0.35080 | 0.36185 |

The largest overall gain was on encounter clusters. Withheld viewpoint remained
the hardest split. The test output preserves all 1,217 hyena rankings and puts
`new_individual` first for 479 hyenas and 619 leopards.

## Run

Set `DATA_ROOT` to the competition files and `CACHE_ROOT` to the saved features.
Run all cells to write the submission. The notebook defaults to cache reuse.
Setting `REBUILD_FEATURES` to `True` enables a separate cold run from the supplied
descriptors. The bundled source includes the metric-learning functions.

In [ ]:
import os
import sys
from pathlib import Path

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
kaggle = Path("/kaggle/input").is_dir()
DATA_ROOT = Path("/kaggle/input") if kaggle else Path.cwd()
if (DATA_ROOT / "projectwild3" / "reid_student_data_main").is_dir():
    DATA_ROOT = DATA_ROOT / "projectwild3"
OUTPUT = Path("/kaggle/working") if kaggle else Path("submission_outputs")
CACHE_ROOT = Path(os.environ.get("WILDLIFE_FEATURE_CACHE", str(OUTPUT / "feature_cache")))
REBUILD_FEATURES = False

from types import ModuleType

notebook_sections = {}


def model_section(line, cell):
    name = line.strip()
    module = ModuleType(f"_wildlife_084751_{name}")
    module.__dict__["notebook_sections"] = notebook_sections
    sys.modules[module.__name__] = module
    exec(compile(cell, f"<notebook section {name}>", "exec"), module.__dict__)
    notebook_sections[name] = module


get_ipython().register_magic_function(model_section, magic_kind="cell")


## Model implementation

All source is defined in the notebook cells below. The sections run in memory. The input paths refer to the competition data and feature caches.


In [ ]:
%%model_section gallery_io
'Load the supplied descriptors and build gallery matches.'
import argparse
from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor
import csv
from dataclasses import dataclass
import hashlib
import json
import os
from pathlib import Path
from threading import Lock
from time import perf_counter
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '4')
import cv2
import numpy as np
from scipy import sparse
from scipy.linalg import eigh
from scipy.special import expit, logit, softmax
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
NEW = 'new_individual'
SPECIES = ('hyena', 'leopard')
ANGLES = {'front': 0, 'frontright': 1, 'right': 2, 'backright': 3, 'back': 4, 'backleft': 5, 'left': 6, 'frontleft': 7}
NAMES = ('v1_cosine', 'v2_global', 'v3_sift')
SHORTLIST = 14
SIFT_IDENTITIES = 9
SIFT_REFS = 2

def unit(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)

def read_table(path):
    with path.open(newline='', encoding='utf-8-sig') as source:
        return list(csv.DictReader(source))

def locate_data(root, need_sift=True):
    root = Path(root)
    required = ('train.csv', 'test.csv', 'miewid_ids.csv', 'miewid_features.npy', 'sample_submission.csv')
    tables = [p for p in [root, root / 'reid_student_data_main'] if all(((p / name).is_file() for name in required))]
    if not tables:
        tables = [p.parent for p in sorted(root.rglob('miewid_ids.csv')) if all(((p.parent / name).is_file() for name in required))]
    if len(tables) != 1:
        raise ValueError(f'Expected one competition table directory under {root}; found {tables}')
    table = tables[0]
    archives = []
    filenames = ('sift_features_train.npz', 'sift_features_test.npz') if need_sift else ()
    for filename in filenames:
        candidates = [table / filename, table.parent / filename, root / filename]
        found = next((p for p in candidates if p.exists()), None)
        if found is None:
            matches = sorted(root.rglob(filename))
            if len(matches) != 1:
                raise ValueError(f'Expected one supplied {filename}; found {matches}')
            found = matches[0]
        archives.append(found)
    return (table, archives)

class Data:

    def __init__(self, root, need_sift=True):
        table, self.archives = locate_data(root, need_sift=need_sift)
        self.train = read_table(table / 'train.csv')
        self.test = read_table(table / 'test.csv')
        self.sample_ids = [row['id'] for row in read_table(table / 'sample_submission.csv')]
        meta = self.train + self.test
        feature_ids = [row['id'] for row in read_table(table / 'miewid_ids.csv')]
        mapping = {identifier: i for i, identifier in enumerate(feature_ids)}
        ids = [row['id'] for row in meta]
        raw = np.load(table / 'miewid_features.npy', allow_pickle=False)
        if len(mapping) != len(feature_ids) or len(set(ids)) != len(ids) or set(feature_ids) != set(ids) or (raw.shape != (len(meta), 2152)) or (not np.isfinite(raw).all()):
            raise ValueError('Unexpected global descriptor shape, IDs, or values')
        if set((row['species'] for row in meta)) != set(SPECIES) or any((not row['identity'] or row['identity'] == NEW for row in self.train)):
            raise ValueError('Unexpected species or empty training identities')
        self.label_species = {row['identity']: row['species'] for row in self.train}
        if any((self.label_species[row['identity']] != row['species'] for row in self.train)):
            raise ValueError('A training identity appears in both species')
        if len(set(self.sample_ids)) != len(self.sample_ids) or set(self.sample_ids) != {row['id'] for row in self.test}:
            raise ValueError('Sample IDs do not match test IDs')
        positions = np.array([mapping[identifier] for identifier in ids])
        self.x = unit(np.ascontiguousarray(raw[positions], dtype=np.float32))
        self.id = np.asarray(ids)
        self.species = np.asarray([row['species'] for row in meta])
        self.view = np.asarray([row['viewpoint'] for row in meta])
        self.angle = np.asarray([ANGLES.get(v, -1) for v in self.view])
        self.label = np.asarray([row.get('identity', '') for row in meta])
        self.train_rows = np.arange(len(self.train))
        self.test_rows = np.arange(len(self.train), len(meta))
        self.duplicates = np.asarray([hashlib.blake2b(v, digest_size=12).hexdigest() for v in raw[positions[:len(self.train)]]])

def split(data, rows, seed, novelty=0.17, known=0.2, cross_view=False):
    """Withhold complete identities for novelty and image groups for known matches."""
    rng = np.random.default_rng(seed)
    gallery, queries = ([], [])
    for species in SPECIES:
        species_rows = rows[data.species[rows] == species]
        labels = data.label[species_rows]
        identities, counts = np.unique(labels, return_counts=True)
        held = set()
        buckets = np.digitize(counts, [2, 4, 8, 16, 32])
        for bucket in np.unique(buckets):
            group = identities[buckets == bucket]
            n = min(len(group) - 1, round(novelty * len(group)))
            held.update(rng.permutation(group)[:n])
        by_identity = {identity: species_rows[labels == identity] for identity in identities}
        for identity, members in by_identity.items():
            if identity in held:
                queries.extend(members)
                continue
            if len(members) < 2 or (len(members) < 5 and rng.random() > 0.65):
                gallery.extend(members)
                continue
            chosen = np.array([], dtype=int)
            if cross_view:
                sides = data.view[members]
                side_choices = [side for side in ('left', 'right') if np.any(sides == side) and np.any(sides != side) and ((sides == side).mean() <= 0.7)]
                if side_choices and rng.random() < 0.7:
                    chosen = members[sides == rng.choice(side_choices)]
            if len(chosen) == 0:
                n = min(len(members) - 1, max(1, round(known * len(members))))
                chosen = rng.permutation(members)[:n]
            mask = np.isin(data.duplicates[members], data.duplicates[chosen])
            if mask.all():
                mask[:] = False
            queries.extend(members[mask])
            gallery.extend(members[~mask])
    return (np.asarray(sorted(gallery)), np.asarray(sorted(queries)))

class SiftStore:

    def __init__(self, data, workers):
        self.ids = data.id
        self.n_train = len(data.train)
        self.archives = [p if p.is_dir() else np.load(p, allow_pickle=False) for p in data.archives]
        self.images = OrderedDict()
        self.pair_cache = {}
        self.lock = Lock()
        self.workers = workers
        cv2.setNumThreads(1)

    def close(self):
        for archive in self.archives:
            if not isinstance(archive, Path):
                archive.close()

    def load(self, row):
        with self.lock:
            if row in self.images:
                self.images.move_to_end(row)
                return self.images[row]
        archive = self.archives[int(row >= self.n_train)]
        keys = [f'{self.ids[row]}_{suffix}' for suffix in ('xy', 'desc')]
        if isinstance(archive, Path):
            xy, desc = [np.load(archive / f'{key}.npy', allow_pickle=False) for key in keys]
        else:
            xy, desc = [archive[key] for key in keys]
        if xy.shape != (len(desc), 2) or desc.ndim != 2 or desc.shape[1] != 128:
            raise ValueError(f'Unexpected SIFT arrays for {self.ids[row]}')
        desc = desc.astype(np.float32)
        norm = desc.sum(axis=1)
        valid = norm > 0
        desc = desc[valid] / norm[valid, None]
        record = (xy[valid].astype(np.float32), np.sqrt(desc, out=desc))
        with self.lock:
            self.images[row] = record
            if len(self.images) > 1400:
                self.images.popitem(last=False)
        return record

    def compare(self, pair):
        xy1, desc1 = self.load(pair[0])
        xy2, desc2 = self.load(pair[1])
        out = np.zeros(6, dtype=np.float32)
        if min(len(desc1), len(desc2)) < 4:
            return out
        out[5] = 1
        similarity = desc1 @ desc2.T
        rows = np.arange(len(desc1))
        nearest = similarity.argmax(axis=1)
        reciprocal = similarity.argmax(axis=0)[nearest] == rows
        d1 = np.maximum(1 - similarity[rows, nearest], 0)
        similarity[rows, nearest] = -np.inf
        d2 = np.maximum(1 - similarity.max(axis=1), 0)
        keep = reciprocal & (d2 > 1e-07) & (d1 < 0.84 ** 2 * d2)
        if keep.sum() < 4:
            return out
        a, b = (xy1[keep], xy2[nearest[keep]])
        ratios = np.sqrt(d1[keep] / d2[keep])
        out[0] = np.log1p(len(a))
        scale = max(float(np.ptp(b, axis=0).max()), 128)
        limit = float(np.clip(0.014 * scale, 3.0, 6.0))
        for fit in (cv2.estimateAffinePartial2D, cv2.estimateAffine2D):
            transform, mask = fit(a, b, method=cv2.RANSAC, ransacReprojThreshold=limit, maxIters=700, confidence=0.995, refineIters=5)
            if transform is None or not np.isfinite(transform).all():
                continue
            stretch = np.linalg.svd(transform[:, :2], compute_uv=False)
            if stretch[1] < 0.18 or stretch[0] > 5 or stretch[0] / stretch[1] > 3.0:
                continue
            inlier = mask.ravel().astype(bool)
            n = int(inlier.sum())
            if n < 4 or np.log1p(n) <= out[1] + 1e-06:
                continue
            extent_a = np.linalg.norm(np.ptp(a[inlier], axis=0))
            extent_b = np.linalg.norm(np.ptp(b[inlier], axis=0))
            out[1:5] = (np.log1p(n), n / len(a), 1 - float(ratios[inlier].mean()), min(extent_a, extent_b) / 512)
        return out

    def features(self, queries, refs):
        q, candidates, count = refs.shape
        result = np.zeros((q, candidates, count, 6), dtype=np.float32)
        requests = {}
        for i, j, k in np.argwhere(refs >= 0):
            key = tuple(sorted((int(queries[i]), int(refs[i, j, k]))))
            requests.setdefault(key, []).append((i, j, k))
        missing = sorted(requests.keys() - self.pair_cache.keys())
        if missing:
            with ThreadPoolExecutor(max_workers=self.workers) as pool:
                self.pair_cache.update(zip(missing, pool.map(self.compare, missing)))
        for key, positions in requests.items():
            for position in positions:
                result[position] = self.pair_cache[key]
        best = result[..., 1].argmax(axis=2)
        return np.concatenate([np.take_along_axis(result, best[:, :, None, None], axis=2).squeeze(2), result[..., 1].mean(axis=2, keepdims=True), result[..., 5].max(axis=2, keepdims=True)], axis=2)

def strongest(matrix, labels, n_labels):
    starts = np.r_[0, np.flatnonzero(np.diff(labels)) + 1]
    if len(starts) != n_labels:
        raise ValueError('Noncontiguous or missing gallery identity')
    return np.maximum.reduceat(matrix, starts, axis=1)

def label_groups(labels):
    return np.split(np.arange(len(labels)), np.flatnonzero(np.diff(labels)) + 1)

def view_affinity(data, rows, cols):
    a, b = (data.angle[rows, None], data.angle[cols][None, :])
    gap = np.minimum(abs(a - b), 8 - abs(a - b))
    valid = (a >= 0) & (b >= 0)
    close = np.where(valid, 1 - gap * 0.25, (data.view[rows, None] == data.view[cols][None, :]) & (data.view[rows, None] != ''))
    return close.astype(np.float32)

def metric_embeddings(x, labels):
    """Regularized within-class whitening and a discriminative linear projection."""
    n = len(labels)
    size = min(144, n - 1, x.shape[1])
    pca = PCA(n_components=size, svd_solver='randomized', random_state=391)
    pca.fit(x[:n])
    projected = pca.transform(x)
    groups = label_groups(labels)
    centers = np.stack([projected[idx].mean(axis=0) for idx in groups])
    residual = projected[:n] - centers[labels]
    within = residual.T @ residual / n
    centered = centers - centers.mean(axis=0)
    between = centered.T @ centered / len(groups)
    within.flat[::size + 1] += 0.9 * np.trace(within) / size + 1e-05
    values, basis = eigh(between, within, check_finite=False)
    weights = np.maximum(values, 0) ** 0.35
    metric = projected @ basis * weights
    return (unit(metric), unit(projected / np.maximum(pca.singular_values_, 1e-05) ** 0.35))

def top(matrix, n):
    n = min(n, matrix.shape[1])
    part = np.argpartition(matrix, -n, axis=1)[:, -n:]
    values = np.take_along_axis(matrix, part, axis=1)
    order = np.argsort(-values, axis=1, kind='stable')
    return np.take_along_axis(part, order, axis=1)

def neighborhood(similarity, k=18):
    cols, vals = ([], [])
    for offset in range(0, len(similarity), 128):
        scores = similarity[offset:offset + 128].copy()
        rows = np.arange(len(scores))
        scores[rows, rows + offset] = -np.inf
        chosen = top(scores, k)
        cols.append(chosen)
        vals.append(np.take_along_axis(scores, chosen, axis=1))
    cols = np.vstack(cols)
    vals = np.vstack(vals)
    weight = np.exp(np.clip((vals - 0.75) / 0.1, -15, 4)).astype(np.float32)
    directed = sparse.csr_matrix((weight.ravel(), (np.repeat(np.arange(len(cols)), cols.shape[1]), cols.ravel())), shape=similarity.shape)
    symmetric = 0.65 * directed.minimum(directed.T) + 0.35 * directed.maximum(directed.T)
    return (symmetric.tocsr(), cols, vals)

def sparse_rowsum(matrix):
    total = np.asarray(matrix.sum(axis=1)).ravel()
    return matrix.multiply(1 / np.maximum(total, 1e-12)[:, None]).tocsr()

def pooled_features(scores, gallery_class, candidates, side, refs, priority):
    out = np.empty((*candidates.shape, 5), dtype=np.float32)
    groups = label_groups(gallery_class)
    for label, gallery_columns in enumerate(groups):
        qi, ci = np.where(candidates == label)
        if len(qi) == 0:
            continue
        sub = np.ix_(qi, gallery_columns)
        scores_for_label = scores[sub]
        best = top(scores_for_label, 3)
        values = np.take_along_axis(scores_for_label, best, axis=1)
        first = best[:, 0]
        out[qi, ci, 0] = values[:, 0]
        out[qi, ci, 1] = values.mean(axis=1)
        out[qi, ci, 2] = values[:, 0] - values[:, -1]
        out[qi, ci, 3] = side[qi, gallery_columns[first]]
        out[qi, ci, 4] = np.log1p(len(gallery_columns))
        active = ci < SIFT_IDENTITIES
        picked = top(priority[sub][active], SIFT_REFS)
        refs[qi[active], ci[active], :picked.shape[1]] = gallery_columns[picked]
    refs[:, 4:, 1] = -1
    secure = (out[:, 0, 0] > 0.67) & (out[:, 0, 0] - np.max(out[:, 1:, 0], axis=1) > 0.18)
    refs[secure, 4:] = -1
    return out

@dataclass
class Episode:
    rows: np.ndarray
    labels: np.ndarray
    species: np.ndarray
    base_labels: np.ndarray
    base_scores: np.ndarray
    candidates: np.ndarray
    global_features: np.ndarray
    query_features: np.ndarray
    sift_features: np.ndarray | None
    gallery_count: np.ndarray
    gallery_has_view: np.ndarray

    @property
    def novel(self):
        return self.labels == NEW

def episode_for_species(data, gallery, query, store=None, transductive=True):
    classes, gal_label = np.unique(data.label[gallery], return_inverse=True)
    order = np.argsort(gal_label, kind='stable')
    gallery, gal_label = (gallery[order], gal_label[order])
    n, nq = (len(gallery), len(query))
    count = len(classes)
    rows = np.r_[gallery, query]
    x = data.x[rows]
    raw = x[n:] @ x[:n].T
    base_values = strongest(raw, gal_label, count)
    base = top(base_values, SHORTLIST)
    centered = unit(x - x[:n].mean(axis=0))
    by_view = centered.copy()
    for view in np.unique(data.view[gallery]):
        use = data.view[gallery] == view
        if view and use.sum() >= 15:
            center = centered[:n][use].mean(axis=0)
            by_view[data.view[rows] == view] -= 0.8 * use.sum() / (use.sum() + 75) * center
    by_view = unit(by_view)
    metric, pca = metric_embeddings(centered, gal_label)
    broad = 0.5 * centered + 0.25 * by_view + 0.25 * x
    weights = np.sqrt(np.array([0.54, 0.2, 0.16, 0.1], dtype=np.float32))
    retrieval = np.concatenate([weight * vectors for weight, vectors in zip(weights, (centered, by_view, x, pca))], axis=1)
    similarity = retrieval @ retrieval.T
    side = view_affinity(data, query, gallery)
    direct = similarity[n:, :n].copy()
    supervised = metric[n:] @ metric[:n].T
    graph, neighbor_cols, neighbor_vals = neighborhood(similarity)
    del similarity, retrieval
    membership = sparse.csr_matrix((np.ones(n, dtype=np.float32), (np.arange(n), gal_label)), shape=(n, count))
    if transductive:
        propagation = sparse_rowsum(graph[n:])
        q_to_g = propagation[:, :n]
        q_to_q = propagation[:, n:]
    else:
        q_to_g = sparse_rowsum(graph[n:, :n])
        q_to_q = sparse.csr_matrix((nq, nq), dtype=np.float32)
    direct_flow = (q_to_g @ membership).toarray()
    flow = direct_flow.copy()
    for _ in range(5):
        flow = direct_flow + 0.83 * (q_to_q @ flow)
    affinity = sparse_rowsum(graph)
    profiles = affinity[n:] @ affinity[:n].T
    shared = profiles.toarray()
    del profiles
    shared /= np.sqrt(np.maximum(np.asarray(affinity[n:].power(2).sum(axis=1)), 1e-09) * np.maximum(np.asarray(affinity[:n].power(2).sum(axis=1)).T, 1e-09))
    pair = 0.86 * direct + 0.14 * supervised + 0.02 * side
    pair_class = strongest(pair, gal_label, count)
    shared_class = strongest(shared, gal_label, count)
    flow_cond = flow / np.maximum(flow.sum(axis=1, keepdims=True), 1e-08)
    combined = pair_class + 0.18 * np.sqrt(flow_cond) + 0.07 * shared_class
    shortlist = np.empty((nq, min(SHORTLIST, count)), dtype=np.int32)
    other = [top(pair_class, 5), top(flow_cond, 4), top(strongest(supervised, gal_label, count), 4), base[:, :4]]
    lead = top(combined, shortlist.shape[1])
    for i in range(nq):
        stream = [*lead[i, :8], *(label for ranks in other for label in ranks[i]), *lead[i, 8:]]
        shortlist[i] = list(dict.fromkeys(stream))[:shortlist.shape[1]]
    row_ids = np.arange(nq)[:, None]
    refs = np.full((*shortlist.shape, SIFT_REFS), -1, dtype=np.int32)
    pooled = pooled_features(pair, gal_label, shortlist, side, refs, pair + 0.06 * side + 0.035 * shared)
    center = np.stack([broad[idx].mean(axis=0) for idx in label_groups(gal_label)])
    prototype = unit(broad[n:]) @ unit(center).T
    global_features = np.stack([base_values[row_ids, shortlist], pair_class[row_ids, shortlist], strongest(direct, gal_label, count)[row_ids, shortlist], strongest(supervised, gal_label, count)[row_ids, shortlist], strongest(pca[n:] @ pca[:n].T, gal_label, count)[row_ids, shortlist], shared_class[row_ids, shortlist], np.sqrt(flow_cond[row_ids, shortlist]), flow[row_ids, shortlist], prototype[row_ids, shortlist], combined[row_ids, shortlist]], axis=2)
    global_features = np.concatenate([global_features, pooled], axis=2)
    global_features = np.concatenate([global_features, global_features[:, :, :10] - global_features[:, :, :10].max(axis=1, keepdims=True)], axis=2)
    query_features = np.column_stack([neighbor_vals[n:, 0], neighbor_vals[n:, -1], (neighbor_cols[n:] >= n).mean(axis=1), flow.sum(axis=1), direct_flow.sum(axis=1), (data.view[query] == '').astype(float), (data.species[query] == 'hyena').astype(float), pair_class.max(axis=1), base_values.max(axis=1), supervised.max(axis=1), shared_class.max(axis=1)]).astype(np.float32)
    counts = np.bincount(gal_label)
    lookup = {label: counts[i] for i, label in enumerate(classes)}
    gallery_count = np.array([lookup.get(label, 0) for label in data.label[query]])
    gallery_views = set(zip(data.label[gallery], data.view[gallery]))
    gallery_has_view = np.array([(label, view) in gallery_views and bool(view) for label, view in zip(data.label[query], data.view[query])])
    truth = np.where(gallery_count > 0, data.label[query], NEW)
    if store is not None:
        refs = np.where(refs >= 0, gallery[refs], -1)
        sift_features = store.features(query, refs)
    else:
        sift_features = None
    return Episode(query, truth, data.species[query], classes[base], base_values[row_ids, base], classes[shortlist], global_features, query_features, sift_features, gallery_count, gallery_has_view)

def stack(pieces):
    return Episode(**{name: None if getattr(pieces[0], name) is None else np.concatenate([getattr(ep, name) for ep in pieces], axis=0) for name in Episode.__dataclass_fields__})

def candidate_inputs(ep, version):
    if version == 1:
        return ep.global_features
    sift = ep.sift_features
    relative = sift[:, :, :2] - sift[:, :, :2].max(axis=1, keepdims=True)
    return np.concatenate([ep.global_features, sift, relative], axis=2)

def classifier(kind, target='candidate'):
    if kind == 'linear':
        c = 0.08 if target == 'candidate' else 0.05
        return make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=500))
    return HistGradientBoostingClassifier(max_iter=105, learning_rate=0.085, max_leaf_nodes=15, min_samples_leaf=100 if target == 'candidate' else 55, l2_regularization=6.0, early_stopping=False, random_state=392)

def model_fit(kind, features, labels, weights, target):
    model = classifier(kind, target)
    if kind == 'linear':
        model.fit(features, labels, logisticregression__sample_weight=weights)
    else:
        model.fit(features, labels, sample_weight=weights)
    return model

def log_odds(model, features):
    return logit(np.clip(model.predict_proba(features)[:, 1], 1e-05, 1 - 1e-05))

def fit_rankers(episode, version, mode):
    if version == 0 or mode == 'distance':
        return []
    x = candidate_inputs(episode, version)
    y = episode.candidates == episode.labels[:, None]
    has_match = y.any(axis=1)
    train_x = x[has_match].reshape(-1, x.shape[2])
    train_y = y[has_match].ravel()
    weights = np.where(train_y, 6.0, 1.0)
    kinds = ['linear'] if mode == 'linear' else ['linear', 'tree']
    return [model_fit(kind, train_x, train_y, weights, 'candidate') for kind in kinds]

def rank(episode, version, models):
    if version == 0:
        return (episode.base_labels, episode.base_scores)
    if not models:
        return (episode.candidates, episode.global_features[:, :, 9])
    x = candidate_inputs(episode, version)
    flat = x.reshape(-1, x.shape[2])
    scores = log_odds(models[0], flat)
    if len(models) == 2:
        scores = 0.45 * scores + 0.55 * log_odds(models[1], flat)
    return (episode.candidates, scores.reshape(x.shape[:2]))

def novelty_inputs(episode, version, scores):
    best = top(scores, 4)
    ordered = np.take_along_axis(scores, best, axis=1)
    first = ordered[:, 0]
    fourth = ordered[:, -1]
    baseline = episode.base_scores
    basic = np.column_stack([baseline[:, 0], baseline[:, 0] - baseline[:, 1], baseline[:, 0] - baseline[:, 3], (episode.species == 'hyena').astype(float)])
    if version == 0:
        return basic
    rows = np.arange(len(scores))
    selected = episode.global_features[rows, best[:, 0]]
    best_signals = episode.global_features[:, :, [1, 3, 5, 6, 7]].max(axis=1)
    features = [basic, episode.query_features, selected[:, [1, 3, 5, 7, 10, 11, 13, 14]], best_signals, np.column_stack([first, first - ordered[:, 1], first - fourth])]
    if version == 2:
        sift = episode.sift_features
        features.extend([sift[rows, best[:, 0]], sift[:, :, :5].max(axis=1)])
    return np.column_stack(features)

def fit_novelty(x, y, species, nonlinear):
    weights = np.ones(len(y))
    for name in SPECIES:
        group = species == name
        for value in (False, True):
            use = group & (y == value)
            weights[use] = 0.5 * group.sum() / max(use.sum(), 1)
    kinds = ['linear', 'tree'] if nonlinear else ['linear']
    return [model_fit(kind, x, y, weights, 'novelty') for kind in kinds]

def temperature(scores, classes, labels):
    match = classes == labels[:, None]
    keep = match.any(axis=1)
    s, correct = (scores[keep], match[keep].argmax(axis=1))
    scale = float(np.median(np.std(s, axis=1)))
    search = np.maximum(scale, 1e-05) * np.geomspace(0.06, 1.3, 25)
    rows = np.arange(len(correct))
    nll = [-np.log(np.maximum(softmax(s / t, axis=1)[rows, correct], 1e-12)).mean() for t in search]
    return float(search[np.argmin(nll)])

@dataclass
class Trained:
    version: int
    mode: str
    rankers: list
    novelty: list
    temperature: float

def fit_variant(episodes, version, mode='linear', nonlinear=False):
    scores, classes, features = ([], [], [])
    for i, episode in enumerate(episodes):
        neighbors = stack([other for j, other in enumerate(episodes) if i != j])
        rankers = fit_rankers(neighbors, version, mode)
        cls, value = rank(episode, version, rankers)
        scores.append(value)
        classes.append(cls)
        features.append(novelty_inputs(episode, version, value))
    data = stack(episodes)
    novelty = fit_novelty(np.concatenate(features), data.novel, data.species, nonlinear)
    temp = temperature(np.concatenate(scores), np.concatenate(classes), data.labels)
    return Trained(version, mode, fit_rankers(data, version, mode), novelty, temp)

def evidence(model, episode):
    labels, values = rank(episode, model.version, model.rankers)
    known = softmax(values / model.temperature, axis=1)
    values_for_novelty = novelty_inputs(episode, model.version, values)
    odds = log_odds(model.novelty[0], values_for_novelty)
    if len(model.novelty) == 2:
        odds = 0.5 * odds + 0.5 * log_odds(model.novelty[1], values_for_novelty)
    return (labels, known, odds)

def estimate_prior(odds, center=0.3, shrink=0.65):
    prior = center
    for _ in range(30):
        update = float(expit(odds + logit(prior)).mean())
        if abs(update - prior) < 0.0001:
            break
        prior = float(np.clip(update, 0.08, 0.7))
    return float(np.clip(center + shrink * (prior - center), 0.12, 0.62))

def write_submission(data, ep, pred, path):
    by_row = dict(zip(data.id[ep.rows], pred))
    output_species = {row['id']: row['species'] for row in data.test}
    if len(by_row) != len(ep.rows) or set(by_row) != set(data.sample_ids):
        raise ValueError('Missing or repeated output IDs')
    for image, labels in by_row.items():
        if not 1 <= len(labels) <= 5 or len(set(labels)) != len(labels):
            raise ValueError(f'Invalid number of distinct guesses at {image}')
        if any((label != NEW and data.label_species.get(label) != output_species[image] for label in labels)):
            raise ValueError(f'Out-of-species prediction at {image}')
    temp = path.with_suffix('.tmp')
    with temp.open('w', newline='', encoding='utf-8') as destination:
        writer = csv.writer(destination, lineterminator='\n')
        writer.writerow(('id', 'predictions'))
        writer.writerows(((image, ' '.join(by_row[image])) for image in data.sample_ids))
    temp.replace(path)


In [ ]:
%%model_section kernel_retrieval
'Build kernel and gallery evidence used by the submission.'
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '2')
os.environ.setdefault('MKL_NUM_THREADS', '1')
import argparse
from dataclasses import fields
from pathlib import Path
from tempfile import TemporaryDirectory
import csv
import json
import numpy as np
from scipy import sparse
from scipy.linalg import cho_factor, cho_solve
from scipy.special import expit, logit, softmax, logsumexp
from scipy.optimize import minimize
r = notebook_sections['gallery_io']

def query_edges(data, q, centered, store):
    n = len(q)
    if n < 2:
        return (np.zeros((n, 1), int), np.zeros((n, 1), np.float32))
    sim = centered @ centered.T
    np.fill_diagonal(sim, -np.inf)
    neighbors = r.top(sim, min(4, n - 1))
    cosine = np.take_along_axis(sim, neighbors, axis=1)
    geo = store.features(q, q[neighbors][:, :, None])
    geometry = expit(4.0 * (geo[:, :, 1] - np.log(9))) * np.clip(geo[:, :, 4] * 6, 0, 1)
    support = np.maximum(expit((cosine - 0.68) / 0.048), geometry)
    return (neighbors, support.astype(np.float32))

def kernel_features(data, gallery, query):
    labels, group = np.unique(data.label[gallery], return_inverse=True)
    x = r.unit(data.x[np.r_[gallery, query]] - data.x[gallery].mean(axis=0))
    n = len(gallery)
    g, q = (x[:n], x[n:])
    gg, qg = (g @ g.T, q @ g.T)
    response = np.eye(len(labels), dtype=np.float32)[group]
    counts = response.sum(axis=0)

    def ridge(k_train, k_query, penalty):
        k_train.flat[::n + 1] += penalty
        return k_query @ cho_solve(cho_factor(k_train, lower=True, check_finite=False), response, check_finite=False)
    linear = ridge(gg.copy(), qg, 0.45)
    polynomial = ridge(0.45 * gg + 0.55 * gg ** 3, 0.45 * qg + 0.55 * qg ** 3, 0.18)
    gg[group[:, None] == group[None, :]] = -2
    impostor_level = np.take_along_axis(gg, r.top(gg, 5), axis=1).mean(axis=1)
    corrected = qg - 0.42 * impostor_level[None, :]
    order = np.argsort(group, kind='stable')
    ordered = group[order]
    direct = r.strongest(qg[:, order], ordered, len(labels))
    local = r.strongest(corrected[:, order], ordered, len(labels))
    centroids = q @ r.unit(response.T @ g / counts[:, None]).T
    density = np.log(np.maximum(np.exp(12 * qg) @ response, 1e-15)) / 12
    density -= 0.5 * np.log(counts)[None, :] / 12
    raw = np.stack([direct, local, linear, polynomial, centroids, density], axis=2)
    leaders = [r.top(raw[:, :, i], 12) for i in (3, 1, 0, 2, 5, 4)]
    width = min(24, len(labels))
    candidates = np.empty((len(q), width), dtype=int)
    for i in range(len(q)):
        sequence = np.stack([item[i] for item in leaders], axis=1).ravel()
        unique = list(dict.fromkeys(sequence))
        if len(unique) < width:
            unique = list(dict.fromkeys([*unique, *np.argsort(-polynomial[i])[:width]]))
        candidates[i] = unique[:width]
    features = raw[np.arange(len(q))[:, None], candidates]
    features = np.concatenate([features, np.log1p(counts[candidates])[:, :, None]], axis=2)
    qs = []
    for i in range(raw.shape[2]):
        v = np.take_along_axis(raw[:, :, i], r.top(raw[:, :, i], 4), axis=1)
        qs.extend([v[:, 0], v[:, 0] - v[:, 1], v[:, 0] - v[:, 3]])
    return (labels[candidates], features, np.column_stack(qs), q)

def prepare_one(data, gallery, query, store, directory, key):
    path = directory / (key + '.npz')
    if path.exists():
        with np.load(path, allow_pickle=False) as saved:
            if not (np.array_equal(saved['gallery_ids'], data.id[gallery]) and np.array_equal(saved['query_ids'], data.id[query])):
                raise ValueError(f'Cache split mismatch for {key}')
        return
    print('Preparing', key, len(gallery), len(query), flush=True)
    base, labels, feats, query_features, links, reliability = ([], [], [], [], [], [])
    offset = 0
    for sp in r.SPECIES:
        g = gallery[data.species[gallery] == sp]
        q = query[data.species[query] == sp]
        base.append(r.episode_for_species(data, g, q, store))
        c, features, signals, centered = kernel_features(data, g, q)
        neighbors, weights = query_edges(data, q, centered, store)
        labels.append(c)
        feats.append(features)
        query_features.append(signals)
        links.append(neighbors + offset)
        reliability.append(weights)
        offset += len(q)
        print(' ', key, sp, 'ready', flush=True)
    ep = r.stack(base)
    values = {f.name: getattr(ep, f.name) for f in fields(ep)}
    temp = path.with_suffix('.partial.npz')
    np.savez_compressed(temp, **values, kernel_labels=np.concatenate(labels), kernel_x=np.concatenate(feats), kernel_q=np.concatenate(query_features), neighbors=np.concatenate(links), weight=np.concatenate(reliability), gallery_ids=data.id[gallery], query_ids=data.id[query])
    temp.replace(path)

def prepare(data, fold_map, store, directory, names):
    directory.mkdir(parents=True, exist_ok=True)
    for name in names:
        prepare_one(data, *fold_map[name], store, directory, name)

def read_episode(directory, key):
    with np.load(directory / (key + '.npz'), allow_pickle=False) as file:
        ep = r.Episode(**{f.name: file[f.name] for f in fields(r.Episode)})
        extras = {k: file[k] for k in ('kernel_labels', 'kernel_x', 'kernel_q', 'neighbors', 'weight')}
    return (ep, extras)

def kernel_novelty(ep, extra):
    idx = extra['neighbors']
    kq = extra['kernel_q']
    best = extra['kernel_labels'][:, 0]
    weights = extra['weight']
    agreement = (best[idx] == best[:, None]) * weights
    return np.column_stack([kq, ep.query_features, weights.max(axis=1), agreement.max(axis=1), (weights - agreement).max(axis=1), ep.sift_features[:, :, 1].max(axis=1), kq[idx, 0].max(axis=1) - kq[:, 0]])

def fit_kernel(episodes):
    x, y, novel_x, novel_y, species = ([], [], [], [], [])
    for ep, extra in episodes:
        hits = extra['kernel_labels'] == ep.labels[:, None]
        keep = hits.any(axis=1)
        x.append(extra['kernel_x'][keep])
        y.append(hits[keep].argmax(axis=1))
        novel_x.append(kernel_novelty(ep, extra))
        novel_y.append(ep.novel)
        species.append(ep.species)
    x, y = (np.concatenate(x).astype(np.float64), np.concatenate(y))
    scale = np.maximum(x.std(axis=(0, 1)), 1e-05)
    x = (x - x.mean(axis=1, keepdims=True)) / scale
    row = np.arange(len(x))

    def objective(coef):
        logits = x @ coef
        loss = (logsumexp(logits, axis=1) - logits[row, y]).mean()
        residual = softmax(logits, axis=1)
        residual[row, y] -= 1
        grad = np.einsum('ij,ijk->k', residual, x) / len(x)
        return (loss + 0.005 * np.dot(coef, coef), grad + 0.01 * coef)
    solution = minimize(objective, np.zeros(x.shape[2]), jac=True, method='L-BFGS-B')
    if not solution.success:
        raise RuntimeError(solution.message)
    novelty = r.fit_novelty(np.concatenate(novel_x), np.concatenate(novel_y), np.concatenate(species), True)
    return (solution.x / scale, novelty)

def fit_experts(episodes):
    base = [ep for ep, _ in episodes]
    print('Fitting V3, global retrieval, and kernel experts', flush=True)
    return (r.fit_variant(base, 2, 'blend', True), r.fit_variant(base, 1, 'distance', False), fit_kernel(episodes))

def dense_posterior(parts, species, vocabulary):
    labels, conditional, odds = parts
    pi = {s: r.estimate_prior(odds[species == s]) for s in r.SPECIES}
    p_new = expit(odds + logit(np.array([pi[s] for s in species])))
    p = np.zeros((len(labels), len(vocabulary) + 1), dtype=np.float64)
    p[np.arange(len(labels))[:, None], np.searchsorted(vocabulary, labels)] = conditional * (1 - p_new[:, None])
    p[:, -1] = p_new
    return p

def expert_probabilities(models, episode, vocabulary):
    ep, extra = episode
    coef, novelty = models[2]
    z = kernel_novelty(ep, extra)
    odds = 0.65 * r.log_odds(novelty[0], z) + 0.35 * r.log_odds(novelty[1], z)
    kernel = (extra['kernel_labels'], softmax(extra['kernel_x'] @ coef, axis=1), odds)
    return [dense_posterior(p, ep.species, vocabulary) for p in (r.evidence(models[0], ep), r.evidence(models[1], ep), kernel)]
RECIPES = {'V3': (0, 0, 0), 'V3 + global': (0.2, 0, 0), 'Kernel expert': (0, 1, 0), 'V3 + kernel 25%': (0, 0.25, 0), 'V3 + kernel 45%': (0, 0.45, 0), 'V3 + verified neighbors': (0, 0, 0.6), 'V3 + kernel + verified neighbors': (0, 0.3, 0.6), 'Three experts + verified neighbors': (0.1, 0.25, 0.72), 'Geometry-gated fusion': (0.05, 0.35, 0.68, True), 'Geometry-gated fusion with stronger propagation': (0.05, 0.4, 0.88, True)}

def ensemble(probabilities, episode, vocabulary, recipe):
    ep, extra = episode
    global_weight, kernel_weight, strength = recipe[:3]
    v3, glob, kernel = probabilities
    initial = (1 - global_weight - kernel_weight) * v3 + global_weight * glob + kernel_weight * kernel
    trust = np.zeros(len(initial))
    if len(recipe) > 3 and recipe[3]:
        geometry = ep.sift_features
        quality = expit(5 * (geometry[:, :, 1] - np.log(10))) * np.clip(geometry[:, :, 4] * 5, 0, 1) * np.clip(geometry[:, :, 2] * 2, 0, 1)
        best = r.top(quality, 2)
        values = np.take_along_axis(quality, best, axis=1)
        trust = 0.7 * (values[:, 0] - values[:, 1]) * values[:, 0]
        match = ep.candidates[np.arange(len(initial)), best[:, 0]]
        initial *= 1 - trust[:, None]
        initial[np.arange(len(initial)), np.searchsorted(vocabulary, match)] += trust
    if not strength:
        return initial
    indices, weights = (extra['neighbors'], extra['weight'])
    reliable = weights ** 3
    graph = sparse.csr_matrix((reliable.ravel(), (np.repeat(np.arange(len(initial)), indices.shape[1]), indices.ravel())), shape=(len(initial), len(initial)))
    graph = r.sparse_rowsum(graph)
    alpha = strength * weights.max(axis=1) ** 1.5 * (1 - trust)
    neighbor_conf = graph @ initial.max(axis=1)
    alpha *= np.clip((neighbor_conf + 0.2) / (initial.max(axis=1) + 0.35), 0.4, 1.2)
    alpha = np.minimum(alpha, 0.95)
    current = initial
    for _ in range(2):
        current = (1 - alpha[:, None]) * initial + alpha[:, None] * (graph @ current)
    return current

def ranked(prob, vocabulary):
    labels = np.r_[vocabulary, np.array([r.NEW])]
    return labels[r.top(prob, 5)]

def validate_inputs(directory, key, fold_map, data):
    episode = read_episode(directory, key)
    ep = episode[0]
    gallery, query = fold_map[key]
    if set(ep.rows) != set(query) or len(ep.rows) != len(query) or np.intersect1d(gallery, query).size:
        raise ValueError(f'Invalid split for {key}')
    if key != 'test' and np.intersect1d(data.duplicates[query], data.duplicates[gallery]).size:
        raise ValueError('Cross-split duplicate')
    return episode

def possible_labels(episode):
    ep, extra = episode
    return np.union1d(ep.candidates, extra['kernel_labels'])


In [ ]:
%%model_section svm_evidence
'Fit the SVM scores used in the spatial branch reference.'
import argparse
import json
from pathlib import Path
ensemble = notebook_sections['kernel_retrieval']
reid = notebook_sections['gallery_io']
import numpy as np
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC
RECIPES = {'V3': ('linear', 0, 0), 'Linear SVM': ('linear', 1, 0), 'RBF SVM': ('rbf', 1, 0), 'RBF SVM novelty': ('rbf', 0, 0.5), 'V3 + 30% linear SVM': ('linear', 0.3, 0), 'V3 + 30% RBF SVM': ('rbf', 0.3, 0), 'V3 + linear SVM and RBF novelty': ('linear', 0.3, 0.35), 'V3 + RBF SVM and RBF novelty': ('rbf', 0.3, 0.35)}

def candidate_features(data, gallery, episode):
    known_views = set(zip(data.label[gallery], data.view[gallery]))
    views = data.view[episode.rows]
    same_view = np.array([[bool(view) and (label, view) in known_views for label in candidates] for view, candidates in zip(views, episode.candidates)], dtype=np.float32)
    x = reid.candidate_inputs(episode, 2)
    return np.concatenate([x, same_view[:, :, None]], axis=2)

def fit_rankers(folds):
    all_x, all_y, all_weights, pairs, pair_weights = ([], [], [], [], [])
    for ep, features in folds:
        correct = ep.labels[:, None] == ep.candidates
        usable = correct.any(axis=1)
        indices = np.flatnonzero(usable)
        x, y = (features[usable], correct[usable])
        weights = np.where(ep.gallery_has_view[indices] | (x[:, :, -1].sum(axis=1) == 0), 1.0, 1.35)
        scores = np.where(y, -np.inf, x[:, :, 9])
        hard = reid.top(scores, 3)
        deep = np.where(y, -np.inf, x[:, :, 5])
        np.put_along_axis(deep, hard, -np.inf, axis=1)
        negatives = np.column_stack([hard, deep.argmax(axis=1)])
        rows = np.arange(len(x))
        right = x[rows, y.argmax(axis=1)]
        wrong = x[rows[:, None], negatives]
        diffs = (right[:, None] - wrong).reshape(-1, x.shape[2])
        pairs.append(diffs)
        pair_weights.append(np.repeat(weights, negatives.shape[1]))
        all_x.append(np.concatenate([right, wrong.reshape(-1, x.shape[2])]))
        all_y.append(np.repeat([True, False], [len(x), len(diffs)]))
        all_weights.append(np.concatenate([2 * weights, np.repeat(weights, negatives.shape[1])]))
    diff = np.concatenate(pairs)
    sample_weight = np.tile(np.concatenate(pair_weights), 2)
    linear = make_pipeline(StandardScaler(with_mean=False), LinearSVC(C=0.003, fit_intercept=False, dual=False, max_iter=4000))
    linear.fit(np.concatenate([diff, -diff]), np.repeat([True, False], len(diff)), linearsvc__sample_weight=sample_weight)
    rbf = make_pipeline(StandardScaler(), SVC(C=1.2, gamma='scale', cache_size=300))
    rbf.fit(np.concatenate(all_x), np.concatenate(all_y), svc__sample_weight=np.concatenate(all_weights))
    return (linear, rbf)

def margins(model, x):
    return model.decision_function(x.reshape(-1, x.shape[2])).reshape(x.shape[:2])

def novelty_weights(y, species):
    weights = np.empty(len(y))
    for sp in reid.SPECIES:
        group = species == sp
        for label in (False, True):
            mask = group & (y == label)
            weights[mask] = group.sum() / (2 * mask.sum())
    return weights

def novelty_svm(sets):
    model = make_pipeline(StandardScaler(), SVC(C=0.8, gamma='scale', cache_size=220))
    x, y, sp = (np.concatenate(c) for c in zip(*sets))
    weights = novelty_weights(y, sp)
    return model.fit(x, y, svc__sample_weight=weights)

def novelty_data(ep, x, rbf_values):
    best = rbf_values.argmax(axis=1)
    views = x[:, :, -1]
    return np.column_stack([reid.novelty_inputs(ep, 2, rbf_values), views[np.arange(len(views)), best], views.mean(axis=1)])

def train_models(folds):
    episodes = [ep for ep, _ in folds]
    print('Fitting V3 and SVMs across', len(episodes), 'calibration episodes', flush=True)
    baseline = reid.fit_variant(episodes, 2, 'blend', True)
    scores = [[], []]
    novelty_sets = []
    for i, (ep, x) in enumerate(folds):
        rivals = fit_rankers([fold for j, fold in enumerate(folds) if i != j])
        values = [margins(model, x) for model in rivals]
        for recorded, score in zip(scores, values):
            recorded.append(score)
        novelty_sets.append((novelty_data(ep, x, values[1]), ep.novel, ep.species))
    labels = np.concatenate([ep.labels for ep in episodes])
    candidates = np.concatenate([ep.candidates for ep in episodes])
    temps = [reid.temperature(np.concatenate(v), candidates, labels) for v in scores]
    decisions = []
    for i, (x, _, _) in enumerate(novelty_sets):
        svm = novelty_svm([fold for j, fold in enumerate(novelty_sets) if i != j])
        decisions.append(svm.decision_function(x))
    y = np.concatenate([ep.novel for ep in episodes])
    species = np.concatenate([ep.species for ep in episodes])
    platt = LogisticRegression(C=2).fit(np.concatenate(decisions)[:, None], y, sample_weight=novelty_weights(y, species))
    return (baseline, fit_rankers(folds), temps, novelty_svm(novelty_sets), platt)

def svm_evidence(fitted, fold):
    ep, x = fold
    baseline, rankers, temps, novelty, platt = fitted
    labels, base_cond, base_odds = reid.evidence(baseline, ep)
    if not np.array_equal(labels, ep.candidates):
        raise ValueError('V3 and the SVMs must use the same candidate ordering')
    svm_scores = [margins(model, x) for model in rankers]
    expert = dict(zip(('linear', 'rbf'), [softmax(v / t, axis=1) for v, t in zip(svm_scores, temps)]))
    new_values = novelty.decision_function(novelty_data(ep, x, svm_scores[1]))
    new_odds = platt.decision_function(new_values[:, None])
    return {name: (labels, (1 - i) * base_cond + i * expert[method], (1 - j) * base_odds + j * new_odds) for name, (method, i, j) in RECIPES.items()}


In [ ]:
%%model_section boosted_reference
'Calibrate the auxiliary spatial reference posterior.'
import argparse
import json
from pathlib import Path
from time import monotonic, sleep
ensemble = notebook_sections['kernel_retrieval']
svm = notebook_sections['svm_evidence']
reid = notebook_sections['gallery_io']
from lightgbm import LGBMClassifier, LGBMRanker
import numpy as np
from scipy.special import softmax
from sklearn.ensemble import HistGradientBoostingClassifier
RECIPES = {'Incumbent': ('blend', 0.65, 0.65, 0.0), 'LightGBM LambdaRank': ('blend', 1.0, 1.0, 1.0), 'Histogram gradient boosting': ('blend', 0.0, 0.0, 1.0), '70% LightGBM + 30% HGB': ('blend', 0.7, 0.7, 1.0), '30% LightGBM + 70% HGB': ('blend', 0.3, 0.3, 1.0), 'LightGBM ranking + HGB novelty': ('blend', 1.0, 0.0, 1.0), 'HGB ranking + LightGBM novelty': ('blend', 0.0, 1.0, 1.0), '75% incumbent + 25% pooled boosting': ('blend', 0.65, 0.65, 0.25), '50% incumbent + 50% pooled boosting': ('blend', 0.65, 0.65, 0.5), '25% incumbent + 75% pooled boosting': ('blend', 0.65, 0.65, 0.75), 'Joint LightGBM ranker with unseen label': ('joint', 0, 0, 1.0), '60% incumbent + 40% joint LightGBM': ('joint', 0, 0, 0.4), 'Boosting for known identities only': ('known', 0.65, 0, 0.45), 'Boosting for novelty only': ('novel', 0, 0.65, 0.5)}
SVM_RECIPE = 'V3 + linear SVM and RBF novelty'
INCUMBENT_KERNEL_RECIPE = 'V3 + kernel 45%'
FOLLOW_UP = {'75% incumbent + 25% selected boosting': ('blend', 1, 0, 0.25), 'Incumbent identities + 30% HGB novelty': ('novel', 0, 0, 0.3), 'View-dependent boosting blend': ('view', 1, 0, 0.35), 'View-dependent HGB novelty': ('view_novel', 0, 0, 0.45)}

def fold(data, folds, cache, key):
    cached = ensemble.validate_inputs(cache, key, folds, data)
    ep, extra = cached
    x = svm.candidate_features(data, folds[key][0], ep)
    matches = ep.candidates[:, :, None] == extra['kernel_labels'][:, None, :]
    ri = np.arange(len(ep.rows))[:, None]
    k = extra['kernel_x'][ri, matches.argmax(axis=2)]
    k = np.where(matches.any(axis=2)[:, :, None], k, np.nan)
    relative = k - np.max(extra['kernel_x'], axis=1, keepdims=True)
    x = np.concatenate([x, k, relative], axis=2)
    geometry = ep.sift_features[:, :, [1, 2, 4]].max(axis=1)
    whole = np.column_stack([ep.query_features, geometry, extra['kernel_q']])
    broadcast = np.broadcast_to(whole[:, None], (*x.shape[:2], whole.shape[1]))
    candidate_x = np.concatenate([x, broadcast], axis=2).astype(np.float32)
    return (cached, candidate_x, whole)

def histogram():
    return HistGradientBoostingClassifier(max_iter=115, learning_rate=0.07, max_leaf_nodes=15, min_samples_leaf=50, l2_regularization=7, early_stopping=False, random_state=885)

def light_model(rank, iterations):
    cls = LGBMRanker if rank else LGBMClassifier
    ranking = {'lambdarank_truncation_level': 6, 'label_gain': [0, 1], 'objective': 'lambdarank'} if rank else {}
    return cls(n_estimators=iterations, learning_rate=0.055, num_leaves=15, min_child_samples=64, reg_lambda=9.0, colsample_bytree=0.85, verbosity=-1, n_jobs=2, random_state=885, **ranking)

def light_predict(model, x):
    return model.booster_.predict(x, raw_score=True, num_threads=2)

def target(ep):
    return ep.candidates == ep.labels[:, None]

def joint_features(item):
    (ep, _), x, whole = item
    blank = np.full((len(x), 1, x.shape[2]), np.nan, dtype=np.float32)
    blank[:, 0, -whole.shape[1]:] = whole
    expanded = np.concatenate([x, blank], axis=1)
    flag = np.zeros((*expanded.shape[:2], 1), dtype=np.float32)
    flag[:, -1, :] = 1
    return np.concatenate([expanded, flag], axis=2)

def fit_rank_models(items):
    x = np.concatenate([a[1] for a in items])
    y = np.concatenate([target(a[0][0]) for a in items])
    valid = y.any(axis=1)
    known_x = x[valid].reshape(-1, x.shape[2])
    hit = y[valid].ravel()
    rank = light_model(True, 145).fit(known_x, hit.astype(int), group=np.full(valid.sum(), x.shape[1]))
    hgb = histogram().fit(known_x, hit, sample_weight=np.where(hit, 6.0, 1.0))
    joint_x = np.concatenate([joint_features(a) for a in items])
    novel = np.concatenate([a[0][0].novel for a in items])
    good = valid | novel
    joint_y = np.column_stack([y, novel])[good]
    joint = light_model(True, 155).fit(joint_x[good].reshape(-1, joint_x.shape[2]), joint_y.ravel().astype(int), group=np.full(good.sum(), joint_x.shape[1]))
    return (rank, hgb, joint)

def raw_scores(models, item):
    _, x, _ = item
    rank, hgb, joint = models
    base = x.reshape(-1, x.shape[2])
    j = joint_features(item)
    return (light_predict(rank, base).reshape(x.shape[:2]), hgb.decision_function(base).reshape(x.shape[:2]), light_predict(joint, j.reshape(-1, j.shape[2])).reshape(j.shape[:2]))

def novelty_vars(item, rank_scores):
    (ep, _), x, whole = item
    direct_parts = [whole]
    for score in rank_scores[:2]:
        direct_parts.append(reid.novelty_inputs(ep, 2, score))
    joint = rank_scores[2]
    direct_parts.extend([joint[:, -1], joint[:, -1] - joint[:, :-1].max(axis=1)])
    return np.column_stack(direct_parts)

def fit_boosting(items):
    out_of_fold = [raw_scores(fit_rank_models([x for j, x in enumerate(items) if j != i]), item) for i, item in enumerate(items)]
    eps = [item[0][0] for item in items]
    x = np.concatenate([novelty_vars(item, scores) for item, scores in zip(items, out_of_fold)])
    novel = np.concatenate([ep.novel for ep in eps])
    species = np.concatenate([ep.species for ep in eps])
    w = svm.novelty_weights(novel, species)
    calibration = (light_model(False, 120).fit(x, novel, sample_weight=w), histogram().fit(x, novel, sample_weight=w))
    labels = np.concatenate([ep.labels for ep in eps])
    candidates = np.concatenate([ep.candidates for ep in eps])
    joint_labels = np.column_stack([candidates, np.full(len(candidates), reid.NEW)])
    temperatures = [reid.temperature(np.concatenate(v), joint_labels if i == 2 else candidates, labels) for i, v in enumerate(zip(*out_of_fold))]
    return (fit_rank_models(items), calibration, temperatures)

def fit_incumbent(items, keys, data, folds):
    episodes = [record[0] for record in items]
    svm_folds = [(ep, svm.candidate_features(data, folds[key][0], ep)) for (ep, _), key in zip(episodes, keys)]
    return (svm.train_models(svm_folds), ensemble.fit_experts(episodes))

def incumbent_posterior(models, item, data, folds, key, vocabulary):
    svm_model, kernel_model = models
    episode, _ = item[0]
    x = svm.candidate_features(data, folds[key][0], episode)
    s = svm.svm_evidence(svm_model, (episode, x))[SVM_RECIPE]
    svm_prob = ensemble.dense_posterior(s, episode.species, vocabulary)
    p = ensemble.expert_probabilities(kernel_model, item[0], vocabulary)
    k = ensemble.ensemble(p, item[0], vocabulary, ensemble.RECIPES[INCUMBENT_KERNEL_RECIPE])
    return 0.5 * svm_prob + 0.5 * k

def predict(boosted, incumbent, item, data, folds, key, choices, *, return_probabilities=False):
    (ep, _), _, _ = item
    vocabulary = ensemble.possible_labels(item[0])
    reference = incumbent_posterior(incumbent, item, data, folds, key, vocabulary)
    models, calibration, temps = boosted
    scores = raw_scores(models, item)
    nv = novelty_vars(item, scores)
    logits = (light_predict(calibration[0], nv), calibration[1].decision_function(nv))
    conditional = [softmax(s / t, axis=1) for s, t in zip(scores[:2], temps[:2])]
    joint = softmax(scores[2] / temps[2], axis=1)
    q0 = np.arange(len(ep.rows))[:, None]
    q1 = np.searchsorted(vocabulary, ep.candidates)
    joint_dense = np.zeros_like(reference)
    joint_dense[q0, q1] = joint[:, :-1]
    joint_dense[:, -1] = joint[:, -1]
    ref_cond = reference[:, :-1] / (1 - reference[:, -1, None]).clip(1e-12)
    gallery = folds[key][0]
    ref_label = vocabulary[ref_cond.argmax(axis=1)]
    views = data.view[ep.rows]
    gallery_views = set(zip(data.label[gallery], data.view[gallery]))
    supported = np.array([bool(view) and (label, view) in gallery_views for label, view in zip(ref_label, views)])
    guesses = {}
    for name in choices:
        settings = FOLLOW_UP[name] if name in FOLLOW_UP else RECIPES[name]
        mode, known_weight, novel_weight, mixing = settings
        if mode == 'joint':
            proposal = joint_dense
        else:
            known = known_weight * conditional[0] + (1 - known_weight) * conditional[1]
            novel = novel_weight * logits[0] + (1 - novel_weight) * logits[1]
            proposal = ensemble.dense_posterior((ep.candidates, known, novel), ep.species, vocabulary)
        if mode == 'known':
            prop_cond = proposal[:, :-1] / (1 - proposal[:, -1, None]).clip(1e-12)
            proposal = np.column_stack([(1 - reference[:, -1, None]) * prop_cond, reference[:, -1]])
        elif mode in ('novel', 'view_novel'):
            proposal = np.column_stack([(1 - proposal[:, -1, None]) * ref_cond, proposal[:, -1]])
        if mode in ('view', 'view_novel'):
            mixing = mixing * np.where(supported, 1.0, 0.25)[:, None]
        full = (1 - mixing) * reference + mixing * proposal
        if not np.isfinite(full).all() or not np.allclose(full.sum(axis=1), 1):
            raise ValueError(f'Invalid probability mass in {name}')
        guesses[name] = full if return_probabilities else ensemble.ranked(full, vocabulary)
    return guesses


In [ ]:
%%model_section reference_posterior
'Reconstruct the auxiliary reference from supplied features.'
import argparse
import json
from pathlib import Path
from time import monotonic, sleep
boost = notebook_sections['boosted_reference']
ensemble = notebook_sections['kernel_retrieval']
reid = notebook_sections['gallery_io']
import numpy as np
from scipy.linalg import cho_factor, cho_solve, eigh
from scipy.special import softmax
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier
CURRENT = 'View-dependent boosting blend'

def side_group(angles):
    return np.where((angles >= 5) & (angles <= 7), -1, np.where((angles >= 1) & (angles <= 3), 1, 0))

def current_probabilities(data, fold_map, cache, training, queries):
    keys = [cache / ('cross_reference_' + name + '.npz') for name in queries]

    def read(path, query):
        with np.load(path, allow_pickle=False) as saved:
            if not np.array_equal(saved['training'], training) or set(saved['query']) != set(data.id[fold_map[query][1]]):
                raise ValueError('Reference posterior cache uses a different calibration split')
            return (saved['posterior'], saved['vocabulary'])
    if all((path.is_file() for path in keys)):
        return [read(path, query) for path, query in zip(keys, queries)]
    print('Refitting current boosting model for', ', '.join(queries), flush=True)
    items = [boost.fold(data, fold_map, cache, key) for key in training]
    current = boost.fit_boosting(items)
    incumbent = boost.fit_incumbent(items, training, data, fold_map)
    result = []
    for path, query in zip(keys, queries):
        item = boost.fold(data, fold_map, cache, query)
        ep = item[0][0]
        v = ensemble.possible_labels(item[0])
        p = boost.predict(current, incumbent, item, data, fold_map, query, [CURRENT], return_probabilities=True)[CURRENT]
        partial = path.with_suffix('.partial.npz')
        np.savez_compressed(partial, posterior=p, vocabulary=v, training=training, query=data.id[ep.rows])
        partial.replace(path)
        result.append((p, v))
    return result


In [ ]:
%%model_section keypoint_io
'Load supplied keypoints and normalize scores.'
from __future__ import annotations
import os
for variable in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[variable] = '2'
import argparse
import csv
import hashlib
import json
import warnings
from concurrent.futures import ThreadPoolExecutor
from contextlib import ExitStack
from dataclasses import dataclass
from pathlib import Path
import cv2
import numpy as np
from scipy import sparse
from scipy.sparse.csgraph import connected_components
from scipy.special import expit, logit, softmax
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def top(matrix, count):
    count = min(count, matrix.shape[1])
    ids = np.argpartition(matrix, -count, axis=1)[:, -count:]
    order = np.argsort(-np.take_along_axis(matrix, ids, axis=1), axis=1, kind='stable')
    return np.take_along_axis(ids, order, axis=1)

def snorm(qg, gg, gallery_labels, k=30):
    gg[gallery_labels[:, None] == gallery_labels] = -np.inf
    g_cohort = np.take_along_axis(gg, top(gg, k), axis=1)
    leaders = gallery_labels[qg.argmax(axis=1)]
    cohort = np.where(gallery_labels[None, :] == leaders[:, None], -np.inf, qg)
    q_cohort = np.take_along_axis(cohort, top(cohort, k), axis=1)
    q = (qg - q_cohort.mean(axis=1, keepdims=True)) / q_cohort.std(axis=1, keepdims=True).clip(0.015)
    g = (qg - g_cohort.mean(axis=1)) / g_cohort.std(axis=1).clip(0.015)
    return 0.5 * (q + g)

class Sift:

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.stack.close()

    def load(self, row):
        archive = self.archives[int(row >= len(self.data.train))]
        image = self.data.id[row]

        def read(suffix):
            key = f'{image}_{suffix}'
            if isinstance(archive, Path):
                return np.load(archive / (key + '.npy'), allow_pickle=False)
            return archive[key]
        xy, desc = (read('xy'), read('desc'))
        valid = desc.sum(axis=1) > 0
        return (xy[valid].astype('f'), desc[valid])


In [ ]:
%%model_section geometry_verification
'Verify supplied keypoint matches with MAGSAC.'
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '2')
os.environ.setdefault('MKL_NUM_THREADS', '1')
import argparse
from collections import OrderedDict, defaultdict
from concurrent.futures import ThreadPoolExecutor
from contextlib import ExitStack
import hashlib
import json
from pathlib import Path
import warnings
import cv2
import numpy as np
from scipy.special import softmax
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
boost = notebook_sections['boosted_reference']
cross = notebook_sections['reference_posterior']
encounter = notebook_sections['keypoint_io']
ensemble = notebook_sections['kernel_retrieval']
reid = notebook_sections['gallery_io']
ROOT = Path.cwd()
CACHE = ROOT / '.wildlife_v3_cache'

def affine_backend():
    x = np.array([[0, 0], [20, 0], [0, 20], [20, 20], [10, 14], [30, 21]], dtype='f')
    try:
        model, _ = cv2.estimateAffinePartial2D(x, x + 1, method=cv2.USAC_MAGSAC)
        if model is not None:
            return 'partial_affine_magsac'
    except cv2.error:
        pass
    return 'full_affine_magsac_projected_to_similarity'

class Verification(encounter.Sift):

    def __init__(self, data, workers, cache):
        self.data = data
        self.workers = workers
        self.stack = ExitStack()
        self.archives = [path if path.is_dir() else self.stack.enter_context(np.load(path, allow_pickle=False)) for path in data.archives]
        self.images = {}
        self.overflow = {}
        self.matches = OrderedDict()
        cv2.setNumThreads(1)
        self.backend = affine_backend()
        tag = hashlib.sha256(data.id.tobytes()).hexdigest()[:16]
        directory = cache / f'rootsift_input_{tag}'
        directory.mkdir(exist_ok=True)
        self.count = self.open_array(directory / 'counts.npy', (len(data.id),), np.int16, -1)
        self.descriptors = self.open_array(directory / 'descriptors.npy', (len(data.id), 500, 128), np.uint8)
        self.positions = self.open_array(directory / 'positions.npy', (len(data.id), 500, 2), np.float32)

    @staticmethod
    def open_array(path, shape, dtype, fill=None):
        if path.exists():
            array = np.load(path, mmap_mode='r+', allow_pickle=False)
            if array.shape != shape or array.dtype != dtype:
                raise ValueError(f'Invalid descriptor cache {path}')
        else:
            array = np.lib.format.open_memmap(path, mode='w+', shape=shape, dtype=dtype)
            if fill is not None:
                array.fill(fill)
                array.flush()
        return array

    def get(self, row):
        if row in self.overflow:
            return self.overflow[row]
        count = int(self.count[row])
        assert count >= 0
        return (self.positions[row, :count], self.descriptors[row, :count])

    def stage_images(self, images, workers):
        self.overflow.clear()
        missing = sorted((row for row in images if self.count[row] < 0))
        for offset in range(0, len(missing), 128):
            batch = missing[offset:offset + 128]
            written = []
            for row, (xy, desc) in zip(batch, workers.map(self.load, batch)):
                count = len(desc)
                compact = desc.astype(np.uint8, copy=False)
                if count > 500 or (desc.dtype != np.uint8 and (not np.array_equal(desc, compact))):
                    self.overflow[row] = (xy, desc)
                    continue
                self.positions[row, :count] = xy
                self.descriptors[row, :count] = compact
                written.append((row, count))
            self.positions.flush()
            self.descriptors.flush()
            for row, count in written:
                self.count[row] = count
            self.count.flush()

    def compare(self, pair):
        xy1, desc1 = self.get(pair[0])
        xy2, desc2 = self.get(pair[1])
        if min(len(desc1), len(desc2)) < 6:
            return (0, 0.0, 0.0)

        def root(matrix):
            return np.sqrt(matrix / matrix.sum(axis=1, keepdims=True, dtype=np.float32))
        sim = root(desc1) @ root(desc2).T
        nearest = sim.argmax(axis=1)
        source = np.arange(len(sim))
        mutual = sim.argmax(axis=0)[nearest] == source
        first = (1 - sim[source, nearest]).clip(0)
        sim[source, nearest] = -2
        second = (1 - sim.max(axis=1)).clip(0)
        keep = mutual & (second > 1e-07) & (first < 0.84 ** 2 * second)
        if keep.sum() < 6:
            return (0, 0.0, 0.0)
        a, b = (xy1[keep], xy2[nearest[keep]])
        limit = float(np.clip(0.014 * max(np.ptp(b, axis=0)), 3, 6))
        results = []
        fit = cv2.estimateAffinePartial2D if self.backend.startswith('partial') else cv2.estimateAffine2D
        transform, mask = fit(a, b, method=cv2.USAC_MAGSAC, ransacReprojThreshold=limit, maxIters=700, confidence=0.995)
        if transform is not None and mask is not None and np.isfinite(transform).all():
            u, s, v = np.linalg.svd(transform[:, :2])
            affine_inliers = mask.ravel().astype(bool)
            if 0.2 <= s[1] <= s[0] <= 4.0 and s[0] / s[1] < 1.5 and (np.linalg.det(u @ v) > 0) and (affine_inliers.sum() >= 6):
                rotation = u @ v * s.mean()
                offset = np.median(b[affine_inliers] - a[affine_inliers] @ rotation.T, axis=0)
                results.append(np.linalg.norm(a @ rotation.T + offset - b, axis=1) <= limit)
        projective, mask = cv2.findHomography(a, b, method=cv2.USAC_MAGSAC, ransacReprojThreshold=limit, maxIters=700, confidence=0.995)
        if projective is not None and mask is not None and np.isfinite(projective).all():
            aa = cv2.perspectiveTransform(a[None], projective)[0]
            try:
                inverse = np.linalg.inv(projective)
            except np.linalg.LinAlgError:
                inverse = None
            span = np.ptp(a, axis=0).clip(1)
            center = np.median(a, axis=0)
            probe = np.vstack([center, center + [0.05 * span[0], 0], center + [0, 0.05 * span[1]]]).astype('f')
            transformed = cv2.perspectiveTransform(probe[None], projective)[0]
            jac = ((transformed[1:] - transformed[0]) / (0.05 * span[:, None])).T
            if inverse is not None and np.isfinite(jac).all() and (np.linalg.det(jac) > 0):
                singular = np.linalg.svd(jac, compute_uv=False)
                if 0.2 <= singular[1] <= singular[0] <= 4.0 and singular[0] / singular[1] < 2.4:
                    bb = cv2.perspectiveTransform(b[None], inverse)[0]
                    results.append(mask.ravel().astype(bool) & (np.linalg.norm(aa - b, axis=1) <= limit) & (np.linalg.norm(bb - a, axis=1) <= limit / singular.mean()))
        best = (0, 0.0, 0.0)
        for mask in results:
            count = int(mask.sum())
            if count < 6:
                continue
            spread = min((np.linalg.norm(np.ptp(x[mask], axis=0)) / max(np.linalg.norm(np.ptp(all_points, axis=0)), 1.0) for x, all_points in ((a, xy1), (b, xy2))))
            best = max(best, (count, count / len(a), float(spread)))
        return best


In [ ]:
%%model_section spatial_features
'Compute spatial texture, PLDA, and candidate features.'
import argparse
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
import hashlib
import json
import os
import sqlite3
anchor = notebook_sections['geometry_verification']
import numpy as np
from scipy import sparse
from scipy.linalg import cho_factor, cho_solve, eigh, solve_triangular
from scipy.special import logsumexp
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.mixture import GaussianMixture
r, e = (anchor.reid, anchor.ensemble)
WIDTH = 24
CACHE = anchor.CACHE / 'animal_v3_v1'
OUT = anchor.ROOT / 'results' / 'animal_v3'
MATRICES = ('cosine', 'pca', 'lda', 'wccn', 'kissme', 'cca', 'view_centroid', 'ridge', 'fisher', 'vlad', 'snorm', 'plda')

def atomic_npz(path, **arrays):
    temporary = path.with_suffix(f'.{os.getpid()}.partial.npz')
    np.savez_compressed(temporary, **arrays)
    temporary.replace(path)

def root(desc):
    x = desc.astype(np.float32)
    return np.sqrt(x / x.sum(axis=1, keepdims=True).clip(1e-12))

def local_array(source, row):
    return source.get(int(row)) if source.count[row] >= 0 else source.load(int(row))

def spatial(xy):
    return xy.astype(np.float32, copy=False) * np.float32(0.25 / 512)

def signed_root_unit(x):
    return r.unit(np.sign(x) * np.sqrt(np.abs(x)))

def texture(source, gallery, query):
    """One PCA and GMM per species and gallery, using up to 40 points per image."""
    rng = np.random.default_rng(3241)
    samples = []
    for row in gallery:
        xy, desc = local_array(source, row)
        chosen = rng.choice(len(desc), min(40, len(desc)), replace=False)
        if len(chosen):
            samples.append(np.column_stack([root(desc[chosen]), spatial(xy[chosen])]))
    training = np.concatenate(samples)
    pca = PCA(n_components=32, svd_solver='randomized', iterated_power=3, random_state=3241)
    reduced = pca.fit_transform(training[:, :128])
    z = np.column_stack([reduced, training[:, 128:]])
    model = GaussianMixture(n_components=32, covariance_type='diag', init_params='k-means++', max_iter=24, tol=0.001, reg_covar=1e-05, random_state=3241)
    model.fit(z)
    rows = np.r_[gallery, query]
    fisher = np.zeros((len(rows), 2 * 32 * 34), dtype=np.float32)
    vlad = np.zeros((len(rows), 32 * 34), dtype=np.float32)
    means, var, prior = (model.means_, model.covariances_, model.weights_)
    for i, row in enumerate(rows):
        xy, desc = local_array(source, row)
        if not len(desc):
            continue
        x = np.column_stack([pca.transform(root(desc)), spatial(xy)])
        resp = model.predict_proba(x)
        mass = resp.sum(axis=0)[:, None]
        first = resp.T @ x
        second = resp.T @ (x * x)
        fv1 = (first - mass * means) / np.sqrt(var * prior[:, None])
        fv2 = ((second - 2 * means * first + mass * means * means) / var - mass) / np.sqrt(2 * prior[:, None])
        fisher[i] = np.r_[fv1.ravel(), fv2.ravel()] / len(x)
        word = resp.argmax(axis=1)
        blocks = np.zeros_like(means)
        np.add.at(blocks, word, x - means[word])
        vlad[i] = r.unit(blocks).ravel()
    info = (len(samples), len(training), bool(model.converged_))
    return (signed_root_unit(fisher), signed_root_unit(vlad), info)

def regularize(covariance, fraction):
    cov = (covariance + covariance.T) * 0.5
    dim = len(cov)
    return (1 - fraction) * cov + (fraction * max(np.trace(cov) / dim, 1e-05) + 1e-06) * np.eye(dim)

def inv_symmetric(matrix):
    return cho_solve(cho_factor(matrix, lower=True, check_finite=False), np.eye(len(matrix)), check_finite=False)

@dataclass
class PLDA:
    mean: np.ndarray
    between: np.ndarray
    within: np.ndarray
    sums: np.ndarray
    counts: np.ndarray

    def class_posteriors(self):
        inv_b, inv_w = (inv_symmetric(self.between), inv_symmetric(self.within))
        for count in np.unique(self.counts):
            cols = np.flatnonzero(self.counts == count)
            cov = inv_symmetric(inv_b + count * inv_w)
            yield (cols, self.sums[cols] @ (inv_w @ cov), cov)

    def llr(self, queries):
        x = queries - self.mean
        marginal = np.linalg.cholesky(self.between + self.within)
        baseline = solve_triangular(marginal, x.T, lower=True)
        baseline = np.sum(baseline ** 2, axis=0) + 2 * np.log(marginal.diagonal()).sum()
        value = np.empty((len(x), len(self.counts)), dtype=np.float64)
        for cols, mean, cov in self.class_posteriors():
            predictive = np.linalg.cholesky(self.within + cov)
            q = solve_triangular(predictive, x.T, lower=True).T
            m = solve_triangular(predictive, mean.T, lower=True).T
            distance = np.sum(q * q, axis=1)[:, None] + np.sum(m * m, axis=1) - 2 * (q @ m.T)
            logdet = 2 * np.log(predictive.diagonal()).sum()
            value[:, cols] = 0.5 * (baseline[:, None] - distance - logdet)
        return value.astype(np.float32)

def fit_plda(gallery, groups, iterations=12):
    counts = np.bincount(groups)
    mean = gallery.mean(axis=0, dtype=np.float64)
    x = gallery.astype(np.float64) - mean
    sums = np.zeros((len(counts), x.shape[1]), dtype=np.float64)
    np.add.at(sums, groups, x)
    centers = sums / counts[:, None]
    residual = x - centers[groups]
    cross = x.T @ x
    between = regularize(0.7 * centers.T @ centers / len(counts), 0.1)
    within = regularize(residual.T @ residual / max(1, len(x) - len(counts)), 0.08)
    initial = PLDA(mean, between, within, sums, counts)
    for _ in range(iterations):
        second_between = np.zeros_like(between)
        second_within = cross.copy()
        for columns, means, cov in initial.class_posteriors():
            count = counts[columns[0]]
            second = means.T @ means + len(columns) * cov
            cross_term = sums[columns].T @ means
            second_between += second
            second_within += count * second - cross_term - cross_term.T
        initial.between = regularize(second_between / len(counts), 0.075)
        initial.within = regularize(second_within / len(x), 0.035)
    return initial

def inverse_root(matrix):
    eig, directions = eigh(matrix, check_finite=False)
    return directions / eig.clip(1e-08) ** 0.5 @ directions.T

def cross_cca(z, group, sides, g_size):
    train_sides = sides[:g_size]
    left, right = ([], [])
    for indices in r.label_groups(group):
        l = indices[train_sides[indices] == -1]
        rr = indices[train_sides[indices] == 1]
        if len(l) and len(rr):
            left.append(z[l].mean(axis=0))
            right.append(z[rr].mean(axis=0))
    if len(left) < 15:
        return z
    l, rr = (np.asarray(left), np.asarray(right))
    lm, rm = (l.mean(0), rr.mean(0))
    l, rr = (l - lm, rr - rm)
    identity = np.eye(z.shape[1]) * 0.09
    a = inverse_root(l.T @ l / len(l) + identity)
    b = inverse_root(rr.T @ rr / len(l) + identity)
    u, d, vt = np.linalg.svd(a @ (l.T @ rr / len(l)) @ b, full_matrices=False)
    k = min(64, len(l) - 1, z.shape[1])
    left_map = a @ u[:, :k] * np.sqrt(d[:k])
    right_map = b @ vt[:k].T * np.sqrt(d[:k])
    result = (z - (lm + rm) * 0.5) @ ((left_map + right_map) * 0.5)
    for side, center, transform in ((-1, lm, left_map), (1, rm, right_map)):
        use = sides == side
        result[use] = (z[use] - center) @ transform
    return r.unit(result).astype(np.float32)

def gallery_metrics(data, g, q, group, counts, raw, fisher, vlad):
    rows = np.r_[g, q]
    n = len(g)
    base = r.unit(np.sign(data.x[rows]) * np.abs(data.x[rows]) ** 0.85)
    pca = PCA(n_components=min(240, n - 1), svd_solver='randomized', iterated_power=3, random_state=888).fit(base[:n])
    projected = pca.transform(base)
    soft = r.unit(projected * (pca.explained_variance_ + 0.0015) ** (-0.29))
    gaussian = projected[:, :64] / np.sqrt(pca.explained_variance_[:64] + 1e-07)
    llr = fit_plda(gaussian[:n], group).llr(gaussian[n:])
    member = np.eye(len(counts), dtype=np.float32)[group]
    centers = member.T @ soft[:n] / counts[:, None]
    residual = soft[:n] - centers[group]
    within = residual.T @ residual / len(residual)
    wccn = r.unit(soft @ inverse_root(within + 0.035 * np.eye(len(within)))).astype(np.float32)
    centered = soft[:n] - soft[:n].mean(0)
    total = centered.T @ centered / len(centered)
    reg = 0.04 * np.eye(len(within))
    eig, basis = eigh(inv_symmetric(2 * within + reg) - inv_symmetric(2 * total + reg), check_finite=False)
    keep = eig > 1e-06
    kissme = r.unit(soft @ (basis[:, keep] * np.sqrt(eig[keep]))).astype(np.float32)
    side = anchor.cross.side_group(data.angle[rows])
    cca = cross_cca(soft[:, :128], group, side, n)
    mask = counts[group] >= 2
    lda = LinearDiscriminantAnalysis(solver='eigen', shrinkage=0.22, n_components=min(144, int((counts >= 2).sum()) - 1))
    lda.fit(soft[:n][mask], group[mask])
    lda_x = r.unit(lda.transform(soft)).astype(np.float32)
    ridge_x = np.column_stack([soft, 0.6 * lda_x])
    cov = ridge_x[:n].T @ ridge_x[:n] + 4 * np.eye(ridge_x.shape[1])
    ridge = cho_solve(cho_factor(cov, lower=True), ridge_x[:n].T @ (member / counts ** 0.7))
    view_centers, view_group = ([], [])
    for position, indices in enumerate(r.label_groups(group)):
        for view in np.unique(data.view[g[indices]]):
            view_centers.append(soft[indices[data.view[g[indices]] == view]].mean(0))
            view_group.append(position)
    norm = anchor.encounter.snorm(raw, data.x[g] @ data.x[g].T, group)

    def reduce(similarity):
        return r.strongest(similarity, group, len(counts))
    similarity = [reduce(raw)]
    similarity.extend((reduce(z[n:] @ z[:n].T) for z in (soft, lda_x, wccn, kissme, cca)))
    similarity.append(r.strongest(soft[n:] @ r.unit(np.asarray(view_centers)).T, np.asarray(view_group), len(counts)))
    similarity.append(ridge_x[n:] @ r.unit(ridge.T).T)
    for z in (fisher, vlad):
        centered_texture = r.unit(z - z[:n].mean(0))
        similarity.append(reduce(centered_texture[n:] @ centered_texture[:n].T))
    return np.stack([*similarity, reduce(norm), llr], axis=2).astype(np.float32)

def rank_indices(similarity):
    centered = similarity - similarity.mean(axis=1, keepdims=True)
    z = centered / similarity.std(axis=1, keepdims=True).clip(1e-05)
    weight = [0.22, 0.12, 0.09, 0.065, 0.065, 0.04, 0.055, 0.09, 0.035, 0.035, 0.065, 0.12]
    fusion = z @ np.asarray(weight, dtype=np.float32)
    main = r.top(fusion, WIDTH)
    specialists = [r.top(similarity[:, :, i], 3) for i in (0, 2, 7, 8, 9, 11)]
    choices = np.empty_like(main)
    for i, ranked in enumerate(main):
        order = [*ranked[:10]]
        for depth in range(3):
            order.extend((channel[i, depth] for channel in specialists))
        choices[i] = list(dict.fromkeys([*order, *ranked[10:]]))[:WIDTH]
    return (choices, fusion)

def select_references(data, g, q, group, chosen, raw):
    refs = np.full((*chosen.shape, 2), -1, dtype=np.int32)
    exact = np.zeros(chosen.shape, dtype=bool)
    for label, indices in enumerate(r.label_groups(group)):
        row, col = np.where(chosen == label)
        score = raw[np.ix_(row, indices)]
        best = score.argmax(1)
        refs[row, col, 0] = g[indices[best]]
        same = (data.view[q[row], None] == data.view[g[indices]][None]) & (data.view[q[row], None] != '')
        exact[row, col] = same.any(1)
        same_best = np.where(same, score, -2).argmax(1)
        use = same.any(1) & (best != same_best)
        refs[row[use], col[use], 1] = g[indices[same_best[use]]]
    assert (refs[:, :, 0] >= 0).all()
    return (refs, exact)

class Geometry:
    """Reuse exact image-pair verification across distinct CV galleries."""

    def __init__(self, source, pool):
        self.source, self.pool = (source, pool)
        self.db = sqlite3.connect(CACHE / 'magsac.sqlite3', timeout=45)
        self.db.execute('PRAGMA busy_timeout=45000')
        self.db.execute('CREATE TABLE IF NOT EXISTS pairs (key INTEGER PRIMARY KEY, n INTEGER, ratio REAL, extent REAL)')
        self.db.commit()

    def scores(self, left, right):
        n = len(self.source.data.id)
        lo, hi = (np.minimum(left, right), np.maximum(left, right))
        codes = lo.astype('int64') * n + hi
        unique, back = np.unique(codes, return_inverse=True)
        known = {}
        for batch in np.array_split(unique, max(1, (len(unique) + 899) // 900)):
            if len(batch):
                table = ','.join(('?' for _ in batch))
                known.update(((key, (count, rat, extent)) for key, count, rat, extent in self.db.execute(f'SELECT key,n,ratio,extent FROM pairs WHERE key IN ({table})', tuple(map(int, batch)))))
        missing = [int(key) for key in unique if key not in known]
        if missing:
            pairs = [divmod(key, n) for key in missing]
            self.source.stage_images({row for pair in pairs for row in pair}, self.pool)
            values = list(self.pool.map(self.source.compare, pairs))
            with self.db:
                self.db.executemany('INSERT OR IGNORE INTO pairs VALUES (?,?,?,?)', ((key, *value) for key, value in zip(missing, values)))
            known.update(zip(missing, values))
        return np.asarray([known[key] for key in unique], dtype=np.float32).reshape(-1, 3)[back]

    def candidates(self, q, refs, path):
        size = len(q)
        done, result = (0, np.zeros((*refs.shape, 3), dtype=np.float32))
        signature = hashlib.sha256(q.tobytes() + refs.tobytes()).hexdigest()
        if path.exists():
            with np.load(path, allow_pickle=False) as stored:
                assert str(stored['signature']) == signature
                done, result = (int(stored['done']), stored['result'])
        for start in range(done, size, 64):
            end = min(start + 64, size)
            photos = refs[start:end]
            ii, jj, kk = np.where(photos >= 0)
            result[start + ii, jj, kk] = self.scores(q[start + ii], photos[ii, jj, kk])
            atomic_npz(path, signature=signature, result=result, done=end)
            if end % 256 == 0 or end == size:
                print(path.stem, end, '/', size, 'queries verified', flush=True)
        return np.stack([result[..., 0].max(2), result[..., 0].sum(2), result[..., 1].max(2), result[..., 2].max(2), (refs >= 0).sum(2)], axis=2).astype(np.float32)

    def graph(self, raw, rows):
        sim = raw @ raw.T
        np.fill_diagonal(sim, -2)
        if len(sim) < 2:
            return sparse.eye(len(sim), format='csr', dtype=np.float32)
        nearest = r.top(sim, min(8, len(sim) - 1))
        left = np.repeat(np.arange(len(rows)), nearest.shape[1])
        right = nearest.ravel()
        mutual = np.any(nearest[right] == left[:, None], axis=1)
        use = (left < right) & mutual & (sim[left, right] >= 0.55)
        left, right = (left[use], right[use])
        cosine = sim[left, right]
        match = self.scores(rows[left], rows[right])[:, 0]
        keep = (cosine >= 0.75) | (match >= 12)
        left, right = (left[keep], right[keep])
        weight = np.maximum(cosine[keep] ** 4, np.minimum(match[keep] / 24, 1))
        graph = sparse.csr_matrix((np.r_[weight, weight], (np.r_[left, right], np.r_[right, left])), shape=(len(rows), len(rows)))
        total = np.asarray(graph.sum(1)).ravel()
        graph += sparse.diags((total == 0).astype('f'))
        return (sparse.diags(1 / np.asarray(graph.sum(1)).ravel()) @ graph).tocsr()

def top_summary(scores):
    values = np.take_along_axis(scores, r.top(scores, 3), axis=1)
    return np.column_stack([values, values[:, 0] - values[:, 1], values[:, 0] - values[:, 2]])

def prep_species(data, key, sp, g, q, geo):
    tag = CACHE / f'{key}_{sp}'
    complete = tag.with_name(tag.name + '_all.npz')
    if complete.exists():
        with np.load(complete, allow_pickle=False) as z:
            assert np.array_equal(z['rows'], q) and np.array_equal(z['gallery'], g)
        return
    g_sorted = g[np.argsort(data.label[g], kind='stable')]
    labels, group, counts = np.unique(data.label[g_sorted], return_inverse=True, return_counts=True)
    draft = tag.with_name(tag.name + '_gallery.npz')
    if not draft.exists():
        print(key, sp, 'galleries', len(g), 'queries', len(q), flush=True)
        fisher, vlad, info = texture(geo.source, g_sorted, q)
        print(key, sp, 'dictionary images, sampled keypoints, EM converged', info, flush=True)
        raw = data.x[q] @ data.x[g_sorted].T
        sims = gallery_metrics(data, g_sorted, q, group, counts, raw, fisher, vlad)
        chosen, fusion = rank_indices(sims)
        refs, exact = select_references(data, g_sorted, q, group, chosen, raw)
        ii = np.arange(len(q))[:, None]
        selected = sims[ii, chosen]
        x = np.concatenate([selected, selected - sims.max(1, keepdims=True), np.stack([np.log1p(counts[chosen]), exact, fusion[ii, chosen]], axis=2)], axis=2)
        novelty = np.column_stack([top_summary(sims[:, :, j]) for j in range(sims.shape[2])] + [logsumexp(sims[:, :, -1], axis=1), exact.mean(1), exact[ii[:, 0], fusion[ii, chosen].argmax(1)], data.angle[q] < 0, np.full(len(q), np.log(len(g_sorted))), np.full(len(q), np.log(len(labels)))])
        atomic_npz(draft, rows=q, gallery=g, labels=labels[chosen], refs=refs, x=x.astype('f'), novelty=novelty.astype('f'), texture_info=info)
    with np.load(draft, allow_pickle=False) as z:
        work = dict(z)
    assert np.array_equal(work['rows'], q) and np.array_equal(work['gallery'], g)
    geometry = geo.candidates(q, work['refs'], tag.with_name(tag.name + '_geometry.npz'))
    graph = geo.graph(data.x[q], q)
    x = np.concatenate([work['x'], geometry, geometry[:, :, :4] - geometry[:, :, :4].max(1, keepdims=True)], axis=2)
    novelty = np.column_stack([work['novelty'], *(top_summary(geometry[:, :, j]) for j in (0, 1, 2, 3))])
    atomic_npz(complete, rows=q, gallery=g, labels=work['labels'], x=x, novelty=novelty, refs=work['refs'], gv=graph.data, gi=graph.indices, gp=graph.indptr, texture_info=work['texture_info'])

def build(names, workers, pack=False, *, data=None, folds=None):
    CACHE.mkdir(parents=True, exist_ok=True)
    if data is None:
        data = r.Data(anchor.ROOT)
    if folds is None:
        raise ValueError("An explicit fold map is required")
    with anchor.Verification(data, workers, anchor.CACHE) as source, ThreadPoolExecutor(max_workers=workers) as pool:
        if pack:
            source.stage_images(set(range(len(data.id))), pool)
            print('SIFT descriptor cache ready', int((source.count >= 0).sum()), flush=True)
        geo = Geometry(source, pool)
        try:
            for name in names:
                gallery, query = folds[name]
                for sp in r.SPECIES:
                    g, q = [rows[data.species[rows] == sp] for rows in (gallery, query)]
                    prep_species(data, name, sp, g, q, geo)
        finally:
            geo.db.close()


In [ ]:
%%model_section spatial_reference
'Align the spatial model reference posterior by photo ID.'
import argparse
from dataclasses import fields
f = notebook_sections['spatial_features']
import numpy as np
a, r = (f.anchor, f.r)

def read(data, rows, name, inner=False):
    path = f.CACHE / f'ref_inner_{name}.npz' if inner else a.CACHE / f'cross_reference_{name}.npz'
    with np.load(path, allow_pickle=False) as z:
        ids = z['ids' if inner else 'query']
        lookup = {image: index for index, image in enumerate(ids)}
        ix = [lookup[name] for name in data.id[rows]]
        return (z['posterior'][ix], z['vocabulary'])


In [ ]:
%%model_section spatial_ranker
'Reproduce the Leopard spatial-model probabilities.'
import argparse
from dataclasses import dataclass
import hashlib
import json
import sys
import warnings
f = notebook_sections['spatial_features']
reference = notebook_sections['spatial_reference']
import numpy as np
from scipy import sparse
from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit, logit, logsumexp, softmax
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from catboost import CatBoostClassifier, CatBoostRanker
from lightgbm import LGBMRanker
from xgboost import XGBRanker
a, r, e = (f.anchor, f.r, f.e)
REFERENCE = '083484_control'
CONTROL = 'clean_v2'
METHOD = 'animal_v3'
FAMILIES = ('lgbm', 'xgb', 'catboost', 'mlp', 'logistic_plus_plda')
RATES = (0.15, 0.3, 0.45, 0.6)

@dataclass
class Block:
    rows: np.ndarray
    truth: np.ndarray
    labels: np.ndarray
    features: np.ndarray
    novelty: np.ndarray
    graph: sparse.csr_matrix
    support: np.ndarray
    refs: np.ndarray
    texture_info: np.ndarray

def load(data, folds, key, species):
    path = f.CACHE / f'{key}_{species}_all.npz'
    with np.load(path, allow_pickle=False) as z:
        gallery, query = [value[data.species[value] == species] for value in folds[key]]
        assert np.array_equal(z['rows'], query) and np.array_equal(z['gallery'], gallery)
        gallery_labels, counts = np.unique(data.label[gallery], return_counts=True)
        true_label = data.label[query]
        index = np.searchsorted(gallery_labels, true_label).clip(max=len(gallery_labels) - 1)
        known = true_label == gallery_labels[index]
        truth = np.where(known, true_label, r.NEW)
        support = np.where(known, counts[index], 0)
        graph = sparse.csr_matrix((z['gv'], z['gi'], z['gp']), shape=(len(query), len(query)))
        return Block(query, truth, z['labels'], z['x'], z['novelty'], graph, support, z['refs'], z['texture_info'])

def relative(features):
    return (features - features.mean(axis=1, keepdims=True)).clip(-150, 150)

def fit_rankers(parts, excluded=()):
    x = np.concatenate([block.features for block in parts])
    truth = np.concatenate([block.truth for block in parts])
    labels = np.concatenate([block.labels for block in parts])
    rows = np.concatenate([block.rows for block in parts])
    match = labels == truth[:, None]
    use = match.any(1) & ~np.isin(rows, excluded)
    x, y = (x[use], match[use])
    width = x.shape[1]
    raw = x.reshape(-1, x.shape[2])
    flat = relative(x).reshape(raw.shape)
    count = np.full(len(x), width)
    group_id = np.repeat(np.arange(len(x)), width)
    light = LGBMRanker(objective='lambdarank', lambdarank_truncation_level=6, label_gain=[0, 1], n_estimators=90, learning_rate=0.065, num_leaves=15, min_child_samples=44, reg_lambda=9, max_bin=45, colsample_bytree=0.88, random_state=725, n_jobs=2, verbosity=-1)
    light.fit(raw, y.ravel().astype(int), group=count)
    xgb = XGBRanker(objective='rank:ndcg', n_estimators=100, learning_rate=0.064, max_depth=4, min_child_weight=9, max_bin=48, colsample_bytree=0.88, subsample=0.92, reg_lambda=11, lambdarank_num_pair_per_sample=7, tree_method='hist', random_state=725, n_jobs=2)
    xgb.fit(raw, y.ravel().astype(int), group=count)
    cat = CatBoostRanker(loss_function='QuerySoftMax', iterations=95, depth=5, border_count=32, learning_rate=0.07, l2_leaf_reg=12, thread_count=2, random_seed=725, verbose=False, allow_writing_files=False)
    cat.fit(raw, y.ravel().astype(int), group_id=group_id)
    mlp = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(26, 12), alpha=0.5, batch_size=160, learning_rate_init=0.004, max_iter=60, random_state=725, n_iter_no_change=8))
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=ConvergenceWarning)
        mlp.fit(flat, y.ravel())
    linear = make_pipeline(StandardScaler(), LogisticRegression(C=0.12, max_iter=400))
    linear.fit(flat, y.ravel(), logisticregression__sample_weight=np.where(y.ravel(), 8.0, 1.0))
    return (light, xgb, cat, mlp, linear)

def rank_scores(models, block):
    raw = block.features.reshape(-1, block.features.shape[2])
    flat = relative(block.features).reshape(raw.shape)
    probs = models[3].predict_proba(flat)[:, 1]
    values = (models[0].booster_.predict(raw, raw_score=True, num_threads=2), models[1].predict(raw), models[2].predict(raw), logit(probs.clip(1e-06, 1 - 1e-06)), models[4].decision_function(flat))
    return np.stack([score.reshape(block.labels.shape) for score in values]).astype(np.float32)

def margins(parts, targets, key, species):
    path = f.CACHE / f'rank_v1_{key}_{species}.npz'
    signature = hashlib.sha256(b''.join((b.rows.tobytes() for b in [*parts, *targets]))).hexdigest()
    if not path.exists():
        scores = []
        for i, b in enumerate(parts):
            partial = path.with_name(path.stem + f'_inner_{i}.npz')
            if partial.exists():
                with np.load(partial, allow_pickle=False) as z:
                    assert str(z['signature']) == signature
                    scores.append(z['margins'])
                continue
            model = fit_rankers([piece for j, piece in enumerate(parts) if j != i], b.rows)
            score = rank_scores(model, b)
            f.atomic_npz(partial, signature=signature, margins=score)
            scores.append(score)
            print(key, species, 'ranker cross-fit', i + 1, flush=True)
        model = fit_rankers(parts)
        scores.extend((rank_scores(model, b) for b in targets))
        f.atomic_npz(path, signature=signature, **{f'score{i}': z for i, z in enumerate(scores)})
    with np.load(path, allow_pickle=False) as z:
        assert str(z['signature']) == signature
        scores = [z[f'score{i}'] for i in range(len(parts) + len(targets))]
    return (scores[:len(parts)], scores[len(parts):])

def conditional_components(b, scores, temps):
    p = softmax(scores / np.asarray(temps[:5])[:, None, None], axis=2)
    plda = softmax(b.features[:, :, len(f.MATRICES) - 1] / temps[5], axis=1)
    p[4] = 0.82 * p[4] + 0.18 * plda
    return p

def track_a_input(block, margins):
    top_margin = [f.top_summary(scores) for scores in margins]
    return np.column_stack([block.novelty, *top_margin])

def fit_novelty(parts, margins, excluded=()):
    x = np.concatenate([track_a_input(b, s) for b, s in zip(parts, margins)])
    y = np.concatenate([b.truth == r.NEW for b in parts])
    rows = np.concatenate([b.rows for b in parts])
    use = ~np.isin(rows, excluded)
    x, y, rows = (x[use], y[use], rows[use])
    _, group, counts = np.unique(rows, return_inverse=True, return_counts=True)
    weight = 1 / counts[group]
    for outcome in (False, True):
        mask = y == outcome
        weight[mask] *= 0.5 * len(y) / weight[mask].sum()
    lr = make_pipeline(StandardScaler(), LogisticRegression(C=0.045, max_iter=500))
    lr.fit(x, y, logisticregression__sample_weight=weight)
    svm = make_pipeline(StandardScaler(), SVC(C=0.85, gamma='scale', probability=True, random_state=3811, cache_size=150))
    svm.fit(x, y, svc__sample_weight=weight)
    hist = HistGradientBoostingClassifier(max_iter=85, learning_rate=0.062, max_leaf_nodes=9, min_samples_leaf=28, l2_regularization=8, early_stopping=False, random_state=3811)
    hist.fit(x, y, sample_weight=weight)
    cat = CatBoostClassifier(iterations=90, learning_rate=0.06, depth=4, border_count=32, l2_leaf_reg=12, random_seed=3811, thread_count=2, verbose=False, allow_writing_files=False)
    cat.fit(x, y, sample_weight=weight)
    return (lr, svm, hist, cat)

def novelty_log_odds(models, b, scores):
    x = track_a_input(b, scores)
    logits = np.stack([logit(m.predict_proba(x)[:, 1].clip(1e-06, 1 - 1e-06)) for m in models])
    return logits.mean(axis=0)

def track_a(b, scores, temps, weights, odds, vocab, prior=0.35):
    cond = np.einsum('m,mqc->qc', weights, conditional_components(b, scores, temps))
    novel = expit(odds + logit(prior))
    posterior = np.zeros((len(b.rows), len(vocab) + 1))
    posterior[np.arange(len(b.rows))[:, None], np.searchsorted(vocab, b.labels)] = cond * (1 - novel[:, None])
    posterior[:, -1] = novel
    return posterior

def track_b(p0, graph, gamma=0.38, iterations=10):
    state = p0
    for _ in range(iterations):
        state = (1 - gamma) * p0 + gamma * (graph @ state)
    return state

def aligned_ref(data, b, name, inner=False):
    old, labels = reference.read(data, b.rows, name, inner)
    vocab = np.union1d(labels, b.labels)
    result = np.zeros((len(b.rows), len(vocab) + 1))
    result[:, np.searchsorted(vocab, labels)] = old[:, :-1]
    result[:, -1] = old[:, -1]
    return (vocab, result)


In [ ]:
%%model_section species_portfolio
'Compute the portfolio known-identity and novelty probabilities.'
import os
from pathlib import Path

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "2"

ON_KAGGLE = Path("/kaggle/input").is_dir()
DATA_ROOT = Path("/kaggle/input") if ON_KAGGLE else Path.cwd()
if not ON_KAGGLE and (DATA_ROOT / "projectwild3" / "reid_student_data_main").is_dir():
    DATA_ROOT = DATA_ROOT / "projectwild3"
OUTPUT = Path("/kaggle/working") if ON_KAGGLE else Path("submission_outputs")
CACHE = OUTPUT / "portfolio_cache"
WORKERS = 5

import sys

import csv
import hashlib
from collections import OrderedDict, defaultdict
from concurrent.futures import ThreadPoolExecutor
from contextlib import ExitStack
from dataclasses import dataclass, fields
from threading import Lock

import cv2
import numpy as np
from scipy import sparse
from scipy.linalg import cho_factor, cho_solve, eigh
from scipy.special import expit, logit, softmax
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, normalize
from catboost import CatBoostRanker
from lightgbm import LGBMRanker
from xgboost import XGBRanker

NEW = "new_individual"
SPECIES = ("hyena", "leopard")
ANGLES = {"front": 0, "frontright": 1, "right": 2, "backright": 3,
          "back": 4, "backleft": 5, "left": 6, "frontleft": 7}
SHORTLIST, SIFT_IDENTITIES, SIFT_REFS, WIDTH = 14, 9, 2, 24


def unit(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)


def read_table(path):
    with path.open(newline="", encoding="utf-8-sig") as source:
        return list(csv.DictReader(source))


def locate_data(root, need_sift=True):
    root = Path(root)
    required = ("train.csv", "test.csv", "miewid_ids.csv", "miewid_features.npy",
                "sample_submission.csv")
    tables = [p for p in [root, root / "reid_student_data_main"]
              if all((p / name).is_file() for name in required)]
    if not tables:
        tables = [p.parent for p in sorted(root.rglob("miewid_ids.csv"))
                  if all((p.parent / name).is_file() for name in required)]
    if len(tables) != 1:
        raise ValueError(f"Expected one competition table directory under {root}; found {tables}")
    table = tables[0]
    archives = []
    filenames = ("sift_features_train.npz", "sift_features_test.npz") if need_sift else ()
    for filename in filenames:
        candidates = [table / filename, table.parent / filename, root / filename]
        found = next((p for p in candidates if p.exists()), None)
        if found is None:
            matches = sorted(root.rglob(filename))
            if len(matches) != 1:
                raise ValueError(f"Expected one supplied {filename}; found {matches}")
            found = matches[0]
        archives.append(found)
    return table, archives


class Data:
    def __init__(self, root, need_sift=True):
        table, self.archives = locate_data(root, need_sift=need_sift)
        self.train = read_table(table / "train.csv")
        self.test = read_table(table / "test.csv")
        self.sample_ids = [row["id"] for row in read_table(table / "sample_submission.csv")]
        meta = self.train + self.test
        feature_ids = [row["id"] for row in read_table(table / "miewid_ids.csv")]
        mapping = {identifier: i for i, identifier in enumerate(feature_ids)}
        ids = [row["id"] for row in meta]
        raw = np.load(table / "miewid_features.npy", allow_pickle=False)
        if (len(mapping) != len(feature_ids) or len(set(ids)) != len(ids)
                or set(feature_ids) != set(ids)
                or raw.shape != (len(meta), 2152)
                or not np.isfinite(raw).all()):
            raise ValueError("Unexpected global descriptor shape, IDs, or values")
        if set(row["species"] for row in meta) != set(SPECIES) or any(
                not row["identity"] or row["identity"] == NEW for row in self.train):
            raise ValueError("Unexpected species or empty training identities")
        self.label_species = {row["identity"]: row["species"] for row in self.train}
        if any(self.label_species[row["identity"]] != row["species"] for row in self.train):
            raise ValueError("A training identity appears in both species")
        if (len(set(self.sample_ids)) != len(self.sample_ids)
                or set(self.sample_ids) != {row["id"] for row in self.test}):
            raise ValueError("Sample IDs do not match test IDs")
        positions = np.array([mapping[identifier] for identifier in ids])
        self.x = unit(np.ascontiguousarray(raw[positions], dtype=np.float32))
        self.id = np.asarray(ids)
        self.species = np.asarray([row["species"] for row in meta])
        self.view = np.asarray([row["viewpoint"] for row in meta])
        self.angle = np.asarray([ANGLES.get(v, -1) for v in self.view])
        self.label = np.asarray([row.get("identity", "") for row in meta])
        self.train_rows = np.arange(len(self.train))
        self.test_rows = np.arange(len(self.train), len(meta))
        self.duplicates = np.asarray([
            hashlib.blake2b(v, digest_size=12).hexdigest() for v in raw[positions[:len(self.train)]]
        ])


def split(data, rows, seed, novelty=0.17, known=0.20, cross_view=False):
    """Withhold complete identities for novelty and image groups for known matches."""
    rng = np.random.default_rng(seed)
    gallery, queries = [], []
    for species in SPECIES:
        species_rows = rows[data.species[rows] == species]
        labels = data.label[species_rows]
        identities, counts = np.unique(labels, return_counts=True)
        held = set()
        buckets = np.digitize(counts, [2, 4, 8, 16, 32])
        for bucket in np.unique(buckets):
            group = identities[buckets == bucket]
            n = min(len(group) - 1, round(novelty * len(group)))
            held.update(rng.permutation(group)[:n])
        by_identity = {
            identity: species_rows[labels == identity] for identity in identities
        }
        for identity, members in by_identity.items():
            if identity in held:
                queries.extend(members)
                continue
            if len(members) < 2 or (len(members) < 5 and rng.random() > 0.65):
                gallery.extend(members)
                continue
            chosen = np.array([], dtype=int)
            if cross_view:
                sides = data.view[members]
                side_choices = [side for side in ("left", "right")
                                if np.any(sides == side) and np.any(sides != side)
                                and (sides == side).mean() <= 0.70]
                if side_choices and rng.random() < 0.70:
                    chosen = members[sides == rng.choice(side_choices)]
            if len(chosen) == 0:
                n = min(len(members) - 1, max(1, round(known * len(members))))
                chosen = rng.permutation(members)[:n]
            # Keep exact duplicate descriptors on the same side of the split.
            mask = np.isin(data.duplicates[members], data.duplicates[chosen])
            if mask.all():
                mask[:] = False
            queries.extend(members[mask])
            gallery.extend(members[~mask])
    return np.asarray(sorted(gallery)), np.asarray(sorted(queries))


class SiftStore:
    def __init__(self, data, workers):
        self.ids = data.id
        self.n_train = len(data.train)
        self.archives = [p if p.is_dir() else np.load(p, allow_pickle=False)
                         for p in data.archives]
        self.images = OrderedDict()
        self.pair_cache = {}
        self.lock = Lock()
        self.workers = workers
        cv2.setNumThreads(1)

    def close(self):
        for archive in self.archives:
            if not isinstance(archive, Path):
                archive.close()

    def load(self, row):
        with self.lock:
            if row in self.images:
                self.images.move_to_end(row)
                return self.images[row]
        archive = self.archives[int(row >= self.n_train)]
        keys = [f"{self.ids[row]}_{suffix}" for suffix in ("xy", "desc")]
        if isinstance(archive, Path):
            xy, desc = [np.load(archive / f"{key}.npy", allow_pickle=False) for key in keys]
        else:
            xy, desc = [archive[key] for key in keys]
        if xy.shape != (len(desc), 2) or desc.ndim != 2 or desc.shape[1] != 128:
            raise ValueError(f"Unexpected SIFT arrays for {self.ids[row]}")
        desc = desc.astype(np.float32)
        norm = desc.sum(axis=1)
        valid = norm > 0
        desc = desc[valid] / norm[valid, None]
        # Hellinger distance on the provided descriptors, with no image processing.
        record = (xy[valid].astype(np.float32), np.sqrt(desc, out=desc))
        with self.lock:
            self.images[row] = record
            if len(self.images) > 1400:
                self.images.popitem(last=False)
        return record

    def compare(self, pair):
        xy1, desc1 = self.load(pair[0])
        xy2, desc2 = self.load(pair[1])
        out = np.zeros(6, dtype=np.float32)
        if min(len(desc1), len(desc2)) < 4:
            return out
        out[5] = 1
        # RootSIFT has unit L2 norm, so 2(1 - dot product) is its squared distance.
        similarity = desc1 @ desc2.T
        rows = np.arange(len(desc1))
        nearest = similarity.argmax(axis=1)
        reciprocal = similarity.argmax(axis=0)[nearest] == rows
        d1 = np.maximum(1 - similarity[rows, nearest], 0)
        similarity[rows, nearest] = -np.inf
        d2 = np.maximum(1 - similarity.max(axis=1), 0)
        keep = reciprocal & (d2 > 1e-7) & (d1 < 0.84 ** 2 * d2)
        if keep.sum() < 4:
            return out
        a, b = xy1[keep], xy2[nearest[keep]]
        ratios = np.sqrt(d1[keep] / d2[keep])
        out[0] = np.log1p(len(a))
        scale = max(float(np.ptp(b, axis=0).max()), 128)
        limit = float(np.clip(0.014 * scale, 3.0, 6.0))
        for fit in (cv2.estimateAffinePartial2D, cv2.estimateAffine2D):
            transform, mask = fit(a, b, method=cv2.RANSAC, ransacReprojThreshold=limit,
                                  maxIters=700, confidence=0.995, refineIters=5)
            if transform is None or not np.isfinite(transform).all():
                continue
            stretch = np.linalg.svd(transform[:, :2], compute_uv=False)
            if stretch[1] < 0.18 or stretch[0] > 5 or stretch[0] / stretch[1] > 3.0:
                continue
            inlier = mask.ravel().astype(bool)
            n = int(inlier.sum())
            if n < 4 or np.log1p(n) <= out[1] + 1e-6:
                continue
            extent_a = np.linalg.norm(np.ptp(a[inlier], axis=0))
            extent_b = np.linalg.norm(np.ptp(b[inlier], axis=0))
            out[1:5] = (
                np.log1p(n), n / len(a), 1 - float(ratios[inlier].mean()),
                min(extent_a, extent_b) / 512,
            )
        return out

    def features(self, queries, refs):
        q, candidates, count = refs.shape
        result = np.zeros((q, candidates, count, 6), dtype=np.float32)
        requests = {}
        for i, j, k in np.argwhere(refs >= 0):
            key = tuple(sorted((int(queries[i]), int(refs[i, j, k]))))
            requests.setdefault(key, []).append((i, j, k))
        missing = sorted(requests.keys() - self.pair_cache.keys())
        if missing:
            with ThreadPoolExecutor(max_workers=self.workers) as pool:
                self.pair_cache.update(zip(missing, pool.map(self.compare, missing)))
        for key, positions in requests.items():
            for position in positions:
                result[position] = self.pair_cache[key]
        best = result[..., 1].argmax(axis=2)
        return np.concatenate([
            np.take_along_axis(result, best[:, :, None, None], axis=2).squeeze(2),
            result[..., 1].mean(axis=2, keepdims=True),
            result[..., 5].max(axis=2, keepdims=True),
        ], axis=2)


def strongest(matrix, labels, n_labels):
    # Gallery columns are ordered by label, so reduceat computes all class maxima.
    starts = np.r_[0, np.flatnonzero(np.diff(labels)) + 1]
    if len(starts) != n_labels:
        raise ValueError("Noncontiguous or missing gallery identity")
    return np.maximum.reduceat(matrix, starts, axis=1)


def label_groups(labels):
    return np.split(np.arange(len(labels)), np.flatnonzero(np.diff(labels)) + 1)


def view_affinity(data, rows, cols):
    a, b = data.angle[rows, None], data.angle[cols][None, :]
    gap = np.minimum(abs(a - b), 8 - abs(a - b))
    valid = (a >= 0) & (b >= 0)
    close = np.where(valid, 1 - gap * 0.25,
                     (data.view[rows, None] == data.view[cols][None, :])
                     & (data.view[rows, None] != ""))
    return close.astype(np.float32)


def metric_embeddings(x, labels):
    """Regularized within-class whitening and a discriminative linear projection."""
    n = len(labels)
    size = min(144, n - 1, x.shape[1])
    pca = PCA(n_components=size, svd_solver="randomized", random_state=391)
    pca.fit(x[:n])
    projected = pca.transform(x)
    groups = label_groups(labels)
    centers = np.stack([projected[idx].mean(axis=0) for idx in groups])
    residual = projected[:n] - centers[labels]
    within = residual.T @ residual / n
    centered = centers - centers.mean(axis=0)
    between = centered.T @ centered / len(groups)
    within.flat[::size + 1] += 0.9 * np.trace(within) / size + 1e-5
    values, basis = eigh(between, within, check_finite=False)
    weights = np.maximum(values, 0) ** 0.35
    metric = (projected @ basis) * weights
    return unit(metric), unit(projected / np.maximum(pca.singular_values_, 1e-5) ** 0.35)


def top(matrix, n):
    n = min(n, matrix.shape[1])
    part = np.argpartition(matrix, -n, axis=1)[:, -n:]
    values = np.take_along_axis(matrix, part, axis=1)
    order = np.argsort(-values, axis=1, kind="stable")
    return np.take_along_axis(part, order, axis=1)


def neighborhood(similarity, k=18):
    cols, vals = [], []
    for offset in range(0, len(similarity), 128):
        scores = similarity[offset:offset + 128].copy()
        rows = np.arange(len(scores))
        scores[rows, rows + offset] = -np.inf
        chosen = top(scores, k)
        cols.append(chosen)
        vals.append(np.take_along_axis(scores, chosen, axis=1))
    cols = np.vstack(cols)
    vals = np.vstack(vals)
    weight = np.exp(np.clip((vals - 0.75) / 0.10, -15, 4)).astype(np.float32)
    directed = sparse.csr_matrix((weight.ravel(), (np.repeat(np.arange(len(cols)), cols.shape[1]),
                                                  cols.ravel())), shape=similarity.shape)
    symmetric = 0.65 * directed.minimum(directed.T) + 0.35 * directed.maximum(directed.T)
    return symmetric.tocsr(), cols, vals


def sparse_rowsum(matrix):
    total = np.asarray(matrix.sum(axis=1)).ravel()
    return matrix.multiply(1 / np.maximum(total, 1e-12)[:, None]).tocsr()


def pooled_features(scores, gallery_class, candidates, side, refs, priority):
    out = np.empty((*candidates.shape, 5), dtype=np.float32)
    groups = label_groups(gallery_class)
    for label, gallery_columns in enumerate(groups):
        qi, ci = np.where(candidates == label)
        if len(qi) == 0:
            continue
        sub = np.ix_(qi, gallery_columns)
        scores_for_label = scores[sub]
        best = top(scores_for_label, 3)
        values = np.take_along_axis(scores_for_label, best, axis=1)
        first = best[:, 0]
        out[qi, ci, 0] = values[:, 0]
        out[qi, ci, 1] = values.mean(axis=1)
        out[qi, ci, 2] = (values[:, 0] - values[:, -1])
        out[qi, ci, 3] = side[qi, gallery_columns[first]]
        out[qi, ci, 4] = np.log1p(len(gallery_columns))
        active = ci < SIFT_IDENTITIES
        picked = top(priority[sub][active], SIFT_REFS)
        refs[qi[active], ci[active], :picked.shape[1]] = gallery_columns[picked]
    # The tail of the shortlist gets one geometric check. For well-separated
    # global leaders, verify the nearest few alternatives rather than every tail.
    refs[:, 4:, 1] = -1
    secure = (out[:, 0, 0] > 0.67) & (
        out[:, 0, 0] - np.max(out[:, 1:, 0], axis=1) > 0.18)
    refs[secure, 4:] = -1
    return out


@dataclass
class Episode:
    rows: np.ndarray
    labels: np.ndarray
    species: np.ndarray
    base_labels: np.ndarray
    base_scores: np.ndarray
    candidates: np.ndarray
    global_features: np.ndarray
    query_features: np.ndarray
    sift_features: np.ndarray | None
    gallery_count: np.ndarray
    gallery_has_view: np.ndarray

    @property
    def novel(self):
        return self.labels == NEW


def episode_for_species(data, gallery, query, store=None, transductive=True):
    classes, gal_label = np.unique(data.label[gallery], return_inverse=True)
    order = np.argsort(gal_label, kind="stable")
    gallery, gal_label = gallery[order], gal_label[order]
    n, nq = len(gallery), len(query)
    count = len(classes)
    rows = np.r_[gallery, query]
    x = data.x[rows]
    raw = x[n:] @ x[:n].T
    base_values = strongest(raw, gal_label, count)
    base = top(base_values, SHORTLIST)
    centered = unit(x - x[:n].mean(axis=0))
    by_view = centered.copy()
    for view in np.unique(data.view[gallery]):
        use = data.view[gallery] == view
        if view and use.sum() >= 15:
            center = centered[:n][use].mean(axis=0)
            by_view[data.view[rows] == view] -= 0.8 * use.sum() / (use.sum() + 75) * center
    by_view = unit(by_view)
    metric, pca = metric_embeddings(centered, gal_label)
    broad = 0.50 * centered + 0.25 * by_view + 0.25 * x
    # Concatenated weighted embeddings produce the average of component similarities.
    weights = np.sqrt(np.array([0.54, 0.20, 0.16, 0.10], dtype=np.float32))
    retrieval = np.concatenate([weight * vectors for weight, vectors
                                in zip(weights, (centered, by_view, x, pca))], axis=1)
    similarity = retrieval @ retrieval.T
    side = view_affinity(data, query, gallery)
    direct = similarity[n:, :n].copy()
    supervised = metric[n:] @ metric[:n].T
    graph, neighbor_cols, neighbor_vals = neighborhood(similarity)
    del similarity, retrieval
    membership = sparse.csr_matrix((np.ones(n, dtype=np.float32),
                                   (np.arange(n), gal_label)), shape=(n, count))
    if transductive:
        propagation = sparse_rowsum(graph[n:])
        q_to_g = propagation[:, :n]
        q_to_q = propagation[:, n:]
    else:
        q_to_g = sparse_rowsum(graph[n:, :n])
        q_to_q = sparse.csr_matrix((nq, nq), dtype=np.float32)
    direct_flow = (q_to_g @ membership).toarray()
    flow = direct_flow.copy()
    for _ in range(5):
        flow = direct_flow + 0.83 * (q_to_q @ flow)
    affinity = sparse_rowsum(graph)
    profiles = affinity[n:] @ affinity[:n].T
    shared = profiles.toarray()
    del profiles
    shared /= np.sqrt(
        np.maximum(np.asarray(affinity[n:].power(2).sum(axis=1)), 1e-9)
        * np.maximum(np.asarray(affinity[:n].power(2).sum(axis=1)).T, 1e-9)
    )
    pair = 0.86 * direct + 0.14 * supervised + 0.02 * side
    pair_class = strongest(pair, gal_label, count)
    shared_class = strongest(shared, gal_label, count)
    flow_cond = flow / np.maximum(flow.sum(axis=1, keepdims=True), 1e-8)
    combined = pair_class + 0.18 * np.sqrt(flow_cond) + 0.07 * shared_class
    # Admit diverse candidates from the learned metric, direct match and graph.
    shortlist = np.empty((nq, min(SHORTLIST, count)), dtype=np.int32)
    other = [top(pair_class, 5), top(flow_cond, 4),
             top(strongest(supervised, gal_label, count), 4), base[:, :4]]
    lead = top(combined, shortlist.shape[1])
    for i in range(nq):
        stream = [*lead[i, :8], *(label for ranks in other for label in ranks[i]), *lead[i, 8:]]
        shortlist[i] = list(dict.fromkeys(stream))[:shortlist.shape[1]]
    row_ids = np.arange(nq)[:, None]
    refs = np.full((*shortlist.shape, SIFT_REFS), -1, dtype=np.int32)
    pooled = pooled_features(pair, gal_label, shortlist, side, refs,
                             pair + 0.06 * side + 0.035 * shared)
    center = np.stack([broad[idx].mean(axis=0) for idx in label_groups(gal_label)])
    prototype = unit(broad[n:]) @ unit(center).T
    global_features = np.stack([
        base_values[row_ids, shortlist],
        pair_class[row_ids, shortlist],
        strongest(direct, gal_label, count)[row_ids, shortlist],
        strongest(supervised, gal_label, count)[row_ids, shortlist],
        strongest(pca[n:] @ pca[:n].T, gal_label, count)[row_ids, shortlist],
        shared_class[row_ids, shortlist],
        np.sqrt(flow_cond[row_ids, shortlist]),
        flow[row_ids, shortlist],
        prototype[row_ids, shortlist],
        combined[row_ids, shortlist],
    ], axis=2)
    global_features = np.concatenate([global_features, pooled], axis=2)
    global_features = np.concatenate([
        global_features, global_features[:, :, :10] - global_features[:, :, :10].max(
            axis=1, keepdims=True),
    ], axis=2)
    query_features = np.column_stack([
        neighbor_vals[n:, 0], neighbor_vals[n:, -1],
        (neighbor_cols[n:] >= n).mean(axis=1),
        flow.sum(axis=1), direct_flow.sum(axis=1),
        (data.view[query] == "").astype(float), (data.species[query] == "hyena").astype(float),
        pair_class.max(axis=1), base_values.max(axis=1),
        supervised.max(axis=1), shared_class.max(axis=1),
    ]).astype(np.float32)
    counts = np.bincount(gal_label)
    lookup = {label: counts[i] for i, label in enumerate(classes)}
    gallery_count = np.array([lookup.get(label, 0) for label in data.label[query]])
    gallery_views = set(zip(data.label[gallery], data.view[gallery]))
    gallery_has_view = np.array([(label, view) in gallery_views and bool(view)
                                 for label, view in zip(data.label[query], data.view[query])])
    truth = np.where(gallery_count > 0, data.label[query], NEW)
    if store is not None:
        refs = np.where(refs >= 0, gallery[refs], -1)
        sift_features = store.features(query, refs)
    else:
        sift_features = None
    return Episode(query, truth, data.species[query], classes[base],
                   base_values[row_ids, base], classes[shortlist],
                   global_features, query_features, sift_features, gallery_count,
                   gallery_has_view)


def stack(pieces):
    return Episode(**{
        name: (None if getattr(pieces[0], name) is None
               else np.concatenate([getattr(ep, name) for ep in pieces], axis=0))
        for name in Episode.__dataclass_fields__
    })


def write_submission(data, ep, pred, path):
    by_row = dict(zip(data.id[ep.rows], pred))
    output_species = {row["id"]: row["species"] for row in data.test}
    if len(by_row) != len(ep.rows) or set(by_row) != set(data.sample_ids):
        raise ValueError("Missing or repeated output IDs")
    for image, labels in by_row.items():
        if not 1 <= len(labels) <= 5 or len(set(labels)) != len(labels):
            raise ValueError(f"Invalid number of distinct guesses at {image}")
        if any(label != NEW and data.label_species.get(label) != output_species[image]
               for label in labels):
            raise ValueError(f"Out-of-species prediction at {image}")
    temp = path.with_suffix(".tmp")
    with temp.open("w", newline="", encoding="utf-8") as destination:
        writer = csv.writer(destination, lineterminator="\n")
        writer.writerow(("id", "predictions"))
        writer.writerows((image, " ".join(by_row[image])) for image in data.sample_ids)
    temp.replace(path)


def calibration_splits(data, gallery):
    return [split(data, gallery, seed, cross_view=(i == 1))
            for i, seed in enumerate((119, 222, 325))]


def query_edges(data, q, centered, store):
    n = len(q)
    if n < 2:
        return np.zeros((n, 1), int), np.zeros((n, 1), np.float32)
    sim = centered @ centered.T
    np.fill_diagonal(sim, -np.inf)
    neighbors = top(sim, min(4, n-1))
    cosine = np.take_along_axis(sim, neighbors, axis=1)
    # Verify query-to-query edges using the same supplied SIFT descriptors.
    geo = store.features(q, q[neighbors][:, :, None])
    geometry = expit(4.0 * (geo[:, :, 1] - np.log(9))) * np.clip(geo[:, :, 4]*6, 0, 1)
    support = np.maximum(expit((cosine - 0.68) / 0.048), geometry)
    return neighbors, support.astype(np.float32)


def kernel_features(data, gallery, query):
    labels, group = np.unique(data.label[gallery], return_inverse=True)
    x = unit(data.x[np.r_[gallery,query]] - data.x[gallery].mean(axis=0))
    n = len(gallery)
    g, q = x[:n], x[n:]
    gg, qg = g @ g.T, q @ g.T
    response = np.eye(len(labels), dtype=np.float32)[group]
    counts = response.sum(axis=0)
    # Fit two independent kernel ridge classifiers on the actual gallery.
    def ridge(k_train, k_query, penalty):
        k_train.flat[::n+1] += penalty
        return k_query @ cho_solve(cho_factor(k_train, lower=True, check_finite=False),
                                   response, check_finite=False)
    linear = ridge(gg.copy(), qg, 0.45)
    polynomial = ridge(0.45*gg + 0.55*gg**3, 0.45*qg + 0.55*qg**3, 0.18)
    # Compare a match against each reference's strongest other-animal matches.
    gg[group[:, None] == group[None, :]] = -2
    impostor_level = np.take_along_axis(gg, top(gg, 5), axis=1).mean(axis=1)
    corrected = qg - 0.42 * impostor_level[None, :]
    order = np.argsort(group, kind="stable")
    ordered = group[order]
    direct = strongest(qg[:, order], ordered, len(labels))
    local = strongest(corrected[:, order], ordered, len(labels))
    centroids = q @ unit((response.T @ g) / counts[:,None]).T
    # Pool several reference photos with a correction for catalogue size.
    density = np.log(np.maximum(np.exp(12*qg) @ response, 1e-15)) / 12
    density -= 0.5*np.log(counts)[None, :]/12
    raw = np.stack([direct, local, linear, polynomial, centroids, density],axis=2)
    # Include the top alternatives from both the supervised and retrieval experts.
    leaders = [top(raw[:,:,i], 12) for i in (3, 1, 0, 2, 5, 4)]
    width = min(24, len(labels))
    candidates = np.empty((len(q),width), dtype=int)
    for i in range(len(q)):
        sequence = np.stack([item[i] for item in leaders], axis=1).ravel()
        unique = list(dict.fromkeys(sequence))
        if len(unique) < width:
            unique = list(dict.fromkeys([*unique, *np.argsort(-polynomial[i])[:width]]))
        candidates[i] = unique[:width]
    features = raw[np.arange(len(q))[:,None], candidates]
    features = np.concatenate([features,
                               np.log1p(counts[candidates])[:,:,None]],axis=2)
    qs = []
    for i in range(raw.shape[2]):
        v = np.take_along_axis(raw[:,:,i], top(raw[:,:,i], 4), axis=1)
        qs.extend([v[:,0],v[:,0]-v[:,1],v[:,0]-v[:,3]])
    return labels[candidates], features, np.column_stack(qs), q


def prepare_one(data, gallery, query, store, directory, key):
    path = directory / (key + ".npz")
    if path.exists():
        with np.load(path, allow_pickle=False) as saved:
            if not (np.array_equal(saved["gallery_ids"], data.id[gallery]) and
                    np.array_equal(saved["query_ids"],data.id[query])):
                raise ValueError(f"Cache split mismatch for {key}")
        return
    print("Preparing", key, len(gallery), len(query), flush=True)
    base, labels, feats, query_features, links, reliability = [], [], [], [], [], []
    offset = 0
    for sp in SPECIES:
        g = gallery[data.species[gallery]==sp]
        q = query[data.species[query]==sp]
        base.append(episode_for_species(data,g,q,store))
        c, features, signals, centered = kernel_features(data,g,q)
        neighbors, weights = query_edges(data,q,centered,store)
        labels.append(c)
        feats.append(features)
        query_features.append(signals)
        links.append(neighbors+offset)
        reliability.append(weights)
        offset += len(q)
        print(" ",key,sp,"ready",flush=True)
    ep = stack(base)
    values = {f.name: getattr(ep,f.name) for f in fields(ep)}
    temp = path.with_suffix(".partial.npz")
    np.savez_compressed(temp, **values, kernel_labels=np.concatenate(labels),
                        kernel_x=np.concatenate(feats), kernel_q=np.concatenate(query_features),
                        neighbors=np.concatenate(links), weight=np.concatenate(reliability),
                        gallery_ids=data.id[gallery], query_ids=data.id[query])
    temp.replace(path)


def prepare(data, fold_map, store, directory, names):
    directory.mkdir(parents=True,exist_ok=True)
    for name in names:
        prepare_one(data,*fold_map[name],store,directory,name)


def read_episode(directory, key):
    with np.load(directory / (key + ".npz"), allow_pickle=False) as file:
        ep = Episode(**{f.name: file[f.name] for f in fields(Episode)})
        extras = {k:file[k] for k in ("kernel_labels","kernel_x","kernel_q","neighbors","weight")}
    return ep, extras


def dense_posterior(parts, species, vocabulary):
    labels, conditional, odds = parts
    pi = {s:estimate_prior(odds[species==s]) for s in SPECIES}
    p_new = expit(odds + logit(np.array([pi[s] for s in species])))
    p = np.zeros((len(labels),len(vocabulary)+1), dtype=np.float64)
    p[np.arange(len(labels))[:,None],np.searchsorted(vocabulary,labels)] = (
        conditional*(1-p_new[:,None]))
    p[:,-1] = p_new
    return p


def ranked(prob, vocabulary):
    labels = np.r_[vocabulary,np.array([NEW])]
    return labels[top(prob,5)]


def validate_inputs(directory,key,fold_map,data):
    episode = read_episode(directory,key)
    ep = episode[0]
    gallery, query = fold_map[key]
    if set(ep.rows) != set(query) or len(ep.rows)!=len(query) or np.intersect1d(gallery,query).size:
        raise ValueError(f"Invalid split for {key}")
    if key!="test" and np.intersect1d(data.duplicates[query],data.duplicates[gallery]).size:
        raise ValueError("Cross-split duplicate")
    return episode


def possible_labels(episode):
    ep, extra = episode
    return np.union1d(ep.candidates, extra["kernel_labels"])


class KeypointSource:

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.stack.close()

    def load(self, row):
        archive = self.archives[int(row >= len(self.data.train))]
        image = self.data.id[row]

        def read(suffix):
            key = f'{image}_{suffix}'
            if isinstance(archive, Path):
                return np.load(archive / (key + '.npy'), allow_pickle=False)
            return archive[key]
        xy, desc = (read('xy'), read('desc'))
        valid = desc.sum(axis=1) > 0
        return (xy[valid].astype('f'), desc[valid])

def affine_backend():
    x = np.array([[0,0],[20,0],[0,20],[20,20],[10,14],[30,21]],dtype="f")
    try:
        model,_ = cv2.estimateAffinePartial2D(x,x+1,method=cv2.USAC_MAGSAC)
        if model is not None:
            return "partial_affine_magsac"
    except cv2.error:
        pass
    return "full_affine_magsac_projected_to_similarity"


class Verification(KeypointSource):
    def __init__(self,data,workers,cache):
        self.data = data
        self.workers = workers
        self.stack = ExitStack()
        self.archives = [path if path.is_dir() else self.stack.enter_context(np.load(path,allow_pickle=False))
                         for path in data.archives]
        self.images = {}
        self.overflow = {}
        self.matches = OrderedDict()
        cv2.setNumThreads(1)
        self.backend = affine_backend()
        tag = hashlib.sha256(data.id.tobytes()).hexdigest()[:16]
        directory = cache / f"rootsift_input_{tag}"
        directory.mkdir(exist_ok=True)
        self.count = self.open_array(directory/"counts.npy",(len(data.id),),np.int16,-1)
        self.descriptors = self.open_array(directory/"descriptors.npy",(len(data.id),500,128),np.uint8)
        self.positions = self.open_array(directory/"positions.npy",(len(data.id),500,2),np.float32)

    @staticmethod
    def open_array(path,shape,dtype,fill=None):
        if path.exists():
            array = np.load(path,mmap_mode="r+",allow_pickle=False)
            if array.shape!=shape or array.dtype!=dtype:
                raise ValueError(f"Invalid descriptor cache {path}")
        else:
            array = np.lib.format.open_memmap(path,mode="w+",shape=shape,dtype=dtype)
            if fill is not None:
                array.fill(fill)
                array.flush()
        return array

    def get(self,row):
        if row in self.overflow:
            return self.overflow[row]
        count = int(self.count[row])
        assert count>=0
        return self.positions[row,:count],self.descriptors[row,:count]

    def stage_images(self,images,workers):
        self.overflow.clear()
        missing = sorted(row for row in images if self.count[row]<0)
        for offset in range(0,len(missing),128):
            batch = missing[offset:offset+128]
            written = []
            for row,(xy,desc) in zip(batch,workers.map(self.load,batch)):
                count = len(desc)
                compact = desc.astype(np.uint8,copy=False)
                if count>500 or (desc.dtype!=np.uint8 and not np.array_equal(desc,compact)):
                    # Preserve atypical arrays exactly in the current batch.
                    self.overflow[row] = xy,desc
                    continue
                self.positions[row,:count] = xy
                self.descriptors[row,:count] = compact
                written.append((row,count))
            self.positions.flush()
            self.descriptors.flush()
            for row,count in written:
                self.count[row] = count
            self.count.flush()

    def compare(self,pair):
        xy1, desc1 = self.get(pair[0])
        xy2, desc2 = self.get(pair[1])
        if min(len(desc1),len(desc2)) < 6:
            return (0,0.,0.)
        def root(matrix):
            return np.sqrt(matrix / matrix.sum(axis=1,keepdims=True,dtype=np.float32))
        sim = root(desc1) @ root(desc2).T
        nearest = sim.argmax(axis=1)
        source = np.arange(len(sim))
        mutual = sim.argmax(axis=0)[nearest] == source
        first = (1-sim[source,nearest]).clip(0)
        sim[source,nearest] = -2
        second = (1-sim.max(axis=1)).clip(0)
        keep = mutual & (second > 1e-7) & (first < .84**2*second)
        if keep.sum()<6:
            return (0,0.,0.)
        a,b = xy1[keep],xy2[nearest[keep]]
        limit = float(np.clip(.014*max(np.ptp(b,axis=0)),3,6))
        results = []
        fit = cv2.estimateAffinePartial2D if self.backend.startswith("partial") else cv2.estimateAffine2D
        transform, mask = fit(a,b,method=cv2.USAC_MAGSAC,ransacReprojThreshold=limit,
                              maxIters=700,confidence=.995)
        if transform is not None and mask is not None and np.isfinite(transform).all():
            u,s,v = np.linalg.svd(transform[:,:2])
            affine_inliers = mask.ravel().astype(bool)
            if .20<=s[1]<=s[0]<=4. and s[0]/s[1]<1.5 and np.linalg.det(u@v)>0 and affine_inliers.sum()>=6:
                rotation = (u@v)*s.mean()
                offset = np.median(b[affine_inliers]-a[affine_inliers]@rotation.T,axis=0)
                results.append(np.linalg.norm(a@rotation.T+offset-b,axis=1)<=limit)
        projective, mask = cv2.findHomography(a,b,method=cv2.USAC_MAGSAC,
                            ransacReprojThreshold=limit,maxIters=700,confidence=.995)
        if projective is not None and mask is not None and np.isfinite(projective).all():
            aa = cv2.perspectiveTransform(a[None],projective)[0]
            # Check the inverse and the local Jacobian to reject unstable warps.
            try:
                inverse = np.linalg.inv(projective)
            except np.linalg.LinAlgError:
                inverse = None
            span = np.ptp(a,axis=0).clip(1)
            center = np.median(a,axis=0)
            probe = np.vstack([center,center+[.05*span[0],0],center+[0,.05*span[1]]]).astype("f")
            transformed = cv2.perspectiveTransform(probe[None],projective)[0]
            jac = ((transformed[1:]-transformed[0])/(.05*span[:,None])).T
            if inverse is not None and np.isfinite(jac).all() and np.linalg.det(jac)>0:
                singular = np.linalg.svd(jac,compute_uv=False)
                if .20<=singular[1]<=singular[0]<=4. and singular[0]/singular[1]<2.4:
                    bb = cv2.perspectiveTransform(b[None],inverse)[0]
                    results.append(mask.ravel().astype(bool) & (np.linalg.norm(aa-b,axis=1)<=limit)
                                   & (np.linalg.norm(bb-a,axis=1)<=limit/singular.mean()))
        best = (0,0.,0.)
        for mask in results:
            count = int(mask.sum())
            if count < 6:
                continue
            spread = min(np.linalg.norm(np.ptp(x[mask],axis=0))/
                         max(np.linalg.norm(np.ptp(all_points,axis=0)),1.)
                         for x,all_points in ((a,xy1),(b,xy2)))
            best = max(best,(count,count/len(a),float(spread)))
        return best

    def features(self,query,gallery,shortlist,candidates,checkpoint):
        qg = gallery[shortlist]
        pairs = [[tuple(sorted((int(q),int(g)))) for g in photos] for q,photos in zip(query,qg)]
        signature = hashlib.sha256(query.tobytes()+qg.tobytes()+candidates.tobytes()).hexdigest()
        result = np.zeros((*candidates.shape,4),dtype="f")
        start = 0
        if checkpoint.exists():
            with np.load(checkpoint,allow_pickle=False) as stored:
                assert str(stored["signature"])==signature
                result = stored["features"]
                start = int(stored["complete"])
        print("Top-50 SIFT",len(query),"queries,",start,"previously completed",flush=True)
        labels = self.data.label[qg]
        with ThreadPoolExecutor(max_workers=self.workers) as workers:
            for offset in range(start,len(query),64):
                end = min(offset+64,len(query))
                todo = sorted({pair for block in pairs[offset:end] for pair in block}-self.matches.keys())
                self.stage_images({row for pair in todo for row in pair},workers)
                for index in range(0,len(todo),256):
                    batch = todo[index:index+256]
                    self.matches.update(zip(batch,workers.map(self.compare,batch)))
                for i in range(offset,end):
                    groups = defaultdict(list)
                    for label,pair in zip(labels[i],pairs[i]):
                        groups[label].append(self.matches[pair])
                    for j,label in enumerate(candidates[i]):
                        if label in groups:
                            values = np.asarray(groups[label])
                            result[i,j] = [values[:,0].max(),np.sort(values[:,0])[-2:].sum(),
                                           values[:,1].max(),values[:,2].max()]
                while len(self.matches)>32000:
                    self.matches.popitem(last=False)
                temp = checkpoint.with_suffix(".partial.npz")
                np.savez_compressed(temp,signature=signature,features=result,complete=end)
                temp.replace(checkpoint)
                if end%256==0 or end==len(query):
                    print("Verified",end,"/",len(query),"queries",flush=True)
        return result


def training_positions(ep):
    eligible = np.flatnonzero((ep.candidates == ep.labels[:,None]).any(axis=1))
    # Positive-known queries are the original identity-ranker training population.
    # Stratify the subset for exact geometry by species and keep all legacy inputs.
    rng = np.random.default_rng(83484)
    positions = []
    for species in SPECIES:
        keep = eligible[ep.species[eligible]==species]
        cap = {"hyena":170,"leopard":300}[species]
        positions.extend(rng.choice(keep,min(len(keep),cap),replace=False))
    return np.sort(positions)

def prepare_geometry(data, folds, key, verifier):
    path = CACHE / f"top50_v1_{key}.npz"
    if path.exists():
        return
    ep, _ = validate_inputs(CACHE, key, folds, data)
    positions = np.arange(len(ep.rows)) if key == "test" else training_positions(ep)
    rows, candidates = ep.rows[positions], ep.candidates[positions]
    gallery = folds[key][0]
    features = np.zeros((*candidates.shape, 7), dtype="f")
    for species in SPECIES:
        take = np.flatnonzero(data.species[rows] == species)
        g = gallery[data.species[gallery] == species]
        g = g[np.argsort(data.label[g], kind="stable")]
        q = rows[take]
        shortlist = top(data.x[q] @ data.x[g].T, 50)
        checkpoint = CACHE / f"top50_v1_{key}_{species}_geometry.npz"
        features[take, :, 3:] = verifier.features(q, g, shortlist, candidates[take], checkpoint)
    signature = hashlib.sha256(data.id[gallery].tobytes() + data.id[rows].tobytes()
                               + candidates.tobytes()).hexdigest()
    temporary = path.with_suffix(".partial.npz")
    np.savez_compressed(temporary, signature=signature, features=features)
    temporary.replace(path)


DICTIONARY_IDS = (
    'tr_00076', 'tr_00123', 'tr_00154', 'tr_00238', 'tr_00273', 'tr_00366', 'tr_00367', 'tr_00399',
    'tr_00493', 'tr_00511', 'tr_00546', 'tr_00580', 'tr_00623', 'tr_01023', 'tr_01157', 'tr_01239',
    'tr_01338', 'tr_01357', 'tr_01395', 'tr_01474', 'tr_01504', 'tr_01566', 'tr_01681', 'tr_01895',
    'tr_01965', 'tr_02004', 'tr_02129', 'tr_02281', 'tr_02316', 'tr_02319', 'tr_02391', 'tr_02508',
    'tr_02595', 'tr_02747', 'tr_02890', 'tr_02961', 'tr_03017', 'tr_03118', 'tr_03135', 'tr_03192',
    'tr_03358', 'tr_03367', 'tr_03437', 'tr_03602', 'tr_03822', 'tr_04034', 'tr_04075', 'tr_04107',
    'tr_04123', 'tr_04294', 'tr_04353', 'tr_04494', 'tr_04545', 'tr_04827', 'tr_04862', 'tr_04977',
    'tr_05038', 'tr_05053', 'tr_05114', 'tr_05129', 'tr_05144', 'tr_05146', 'tr_05225', 'tr_05314',
    'tr_05354', 'tr_05369', 'tr_05370', 'tr_05439', 'tr_05456', 'tr_05462', 'tr_05464', 'tr_05533',
    'tr_05560', 'tr_05647', 'tr_05825', 'tr_05863', 'tr_05888', 'tr_05927', 'tr_05969', 'tr_05970',
    'tr_05982', 'tr_06004', 'tr_06019', 'tr_06069', 'tr_06256', 'tr_06326', 'tr_06469', 'tr_06541',
    'tr_06667', 'tr_06859', 'tr_06937', 'tr_07004', 'tr_07026', 'tr_07123', 'tr_07136', 'tr_07147',
)

def root(desc):
    desc = desc.astype(np.float32)
    return np.sqrt(desc/desc.sum(axis=1,keepdims=True).clip(1e-12))


def encode(desc,pca,mix):
    x = pca.transform(root(desc))
    q = mix.predict_proba(x)
    diff = (x[:,None,:]-mix.means_[None])/np.sqrt(mix.covariances_[None])
    weight = q[:,:,None]
    a = (weight*diff).mean(0)/np.sqrt(mix.weights_[:,None])
    b = (weight*(diff**2-1)).mean(0)/np.sqrt(2*mix.weights_[:,None])
    hard = q.argmax(1)
    vlad = np.zeros_like(mix.means_)
    np.add.at(vlad,hard,x-mix.means_[hard])
    vlad = unit(vlad)
    def normalize(z):
        z = np.sign(z)*np.sqrt(np.abs(z))
        return z/max(np.linalg.norm(z),1e-12)
    return normalize(np.r_[a.ravel(),b.ravel()]),normalize(vlad.ravel())

def texture_codes(data, folds, keys, verifier):
    path = CACHE / "amigo_texture_v1.npz"
    index = {name: i for i, name in enumerate(data.id)}
    common = np.array([index[name] for name in DICTIONARY_IDS])
    for key in keys:
        if not np.isin(common, folds[key][0]).all():
            raise ValueError("Texture dictionary rows must belong to the gallery")
    if not path.exists():
        rng = np.random.default_rng(642)
        def load(row):
            return verifier.get(row)[1] if verifier.count[row] >= 0 else verifier.load(row)[1]
        samples = []
        for row in common:
            desc = load(row)
            if len(desc):
                samples.append(root(desc[rng.choice(len(desc), min(len(desc), 180), replace=False)]))
        local = np.concatenate(samples)
        pca = PCA(n_components=32, svd_solver="randomized", random_state=642)
        z = pca.fit_transform(local)
        mix = GaussianMixture(n_components=24, covariance_type="diag", max_iter=55,
                              reg_covar=1e-5, random_state=642).fit(z)
        fisher = np.zeros((len(data.id),1536), dtype="float16")
        vlad = np.zeros((len(data.id),768), dtype="float16")
        for row in range(len(data.id)):
            desc = load(row)
            if len(desc):
                fisher[row], vlad[row] = encode(desc, pca, mix)
        np.savez(path, ids=data.id, fisher=fisher, vlad=vlad, dictionary_ids=data.id[common])
    with np.load(path, allow_pickle=False) as cache:
        np.testing.assert_array_equal(cache["ids"], data.id)
        np.testing.assert_array_equal(cache["dictionary_ids"], DICTIONARY_IDS)
        return cache["fisher"], cache["vlad"]


def fit_wccn(gallery, groups, query, reg=.06, power=.50):
    residuals = np.vstack([gallery[idx] - gallery[idx].mean(axis=0, keepdims=True)
                           for idx in groups if len(idx) >= 2])
    values, basis = np.linalg.eigh((residuals.T @ residuals) / float(len(residuals)))
    projection = (basis * ((np.maximum(0.0, values) + reg) ** -power)[None, :]) @ basis.T
    return (normalize(gallery @ projection, axis=1).astype(np.float32),
            normalize(query @ projection, axis=1).astype(np.float32))


def fit_kissme(gallery, groups, query, reg_s=.12, reg_d=.12, max_dim=256):
    d = min(max_dim, gallery.shape[1])
    g = normalize(gallery[:, -d:], axis=1).astype(np.float64)
    q = normalize(query[:, -d:], axis=1).astype(np.float64)
    residuals = []
    for idx in groups:
        count = len(idx)
        if count >= 2:
            examples = g[idx]
            factor = np.sqrt(float(count)) / float(count)
            residuals.append((examples - examples.mean(0, keepdims=True)) * np.sqrt(factor * count))
    joined = np.vstack(residuals)
    same_covariance = 2.0 * (joined.T @ joined) / float(len(joined))
    centered = g - g.mean(0, keepdims=True)
    other_covariance = 2.0 * (centered.T @ centered) / float(len(g))
    matrix = np.linalg.inv(same_covariance + reg_s * np.eye(d)) - np.linalg.inv(other_covariance + reg_d * np.eye(d))
    values, basis = np.linalg.eigh(0.5 * (matrix + matrix.T))
    positive = values > 1e-6
    projection = (basis[:, positive] * np.sqrt(values[positive])[None, :] if positive.sum() >= 16
                  else basis[:, -min(64,d):])
    return normalize(g @ projection, axis=1).astype(np.float32), normalize(q @ projection, axis=1).astype(np.float32)


def fit_cross_view_cca(gallery, groups, gallery_views, query, query_views, reg=.18, max_dim=224, n_comp=96):
    left = ("left", "frontleft", "backleft")
    right = ("right", "frontright", "backright")
    d = min(max_dim, gallery.shape[1])
    g = normalize(gallery[:, -d:], axis=1).astype(np.float64)
    q = normalize(query[:, -d:], axis=1).astype(np.float64)
    paired = []
    for idx in groups:
        li = idx[np.isin(gallery_views[idx], left)]
        ri = idx[np.isin(gallery_views[idx], right)]
        if len(li) and len(ri):
            paired.append((g[li].mean(axis=0), g[ri].mean(axis=0)))
    if len(paired) < 20:
        return gallery[:, -n_comp:].astype(np.float32), query[:, -n_comp:].astype(np.float32)
    l, r = [np.vstack(side) for side in zip(*paired)]
    mean_l, mean_r = l.mean(0, keepdims=True), r.mean(0, keepdims=True)
    l, r = l - mean_l, r - mean_r
    count = float(len(l))
    def inverse_root(samples):
        values, basis = np.linalg.eigh((samples.T @ samples) / count + reg * np.eye(d))
        return (basis * (np.maximum(values, 1e-8) ** -0.5)[None, :]) @ basis.T
    norm_l, norm_r = inverse_root(l), inverse_root(r)
    u, correlation, vt = np.linalg.svd(norm_l @ ((l.T @ r) / count) @ norm_r, full_matrices=False)
    k = min(n_comp, len(correlation))
    weight = np.sqrt(np.clip(correlation[:k], 0.0, 1.0))[None, :]
    proj_l = (norm_l @ u[:, :k]) * weight
    proj_r = (norm_r @ vt[:k, :].T) * weight
    middle = 0.5 * (proj_l + proj_r)
    mean = 0.5 * (mean_l + mean_r)
    def project(x, views):
        out = (x - mean) @ middle
        for side, center, projection in ((left, mean_l, proj_l), (right, mean_r, proj_r)):
            use = np.isin(views, side)
            if np.any(use):
                out[use] = (x[use] - center) @ projection
        return normalize(out, axis=1).astype(np.float32)
    return project(g, gallery_views), project(q, query_views)


def candidate_features(data, gallery, episode):
    gallery_views = set(zip(data.label[gallery], data.view[gallery]))
    supported = np.array([[bool(view) and (label, view) in gallery_views for label in labels]
                          for view, labels in zip(data.view[episode.rows], episode.candidates)], dtype=np.float32)
    sift = episode.sift_features
    relative = sift[:, :, :2] - sift[:, :, :2].max(axis=1, keepdims=True)
    return np.concatenate([episode.global_features, sift, relative, supported[:, :, None]], axis=2)


def light_model(iterations):
    return LGBMRanker(n_estimators=iterations, learning_rate=.055, num_leaves=15,
                      min_child_samples=64, reg_lambda=9.0, colsample_bytree=.85,
                      verbosity=-1, n_jobs=2, random_state=885,
                      lambdarank_truncation_level=6, label_gain=[0,1], objective="lambdarank")


def retrieval_fold(data, folds, cache, key):
    cached = validate_inputs(cache, key, folds, data)
    ep, extra = cached
    x = candidate_features(data, folds[key][0], ep)
    matches = ep.candidates[:, :, None] == extra["kernel_labels"][:, None, :]
    ri = np.arange(len(ep.rows))[:, None]
    k = extra["kernel_x"][ri, matches.argmax(axis=2)]
    k = np.where(matches.any(axis=2)[:, :, None], k, np.nan)
    relative = k - np.max(extra["kernel_x"], axis=1, keepdims=True)
    x = np.concatenate([x, k, relative], axis=2)
    # Per-query evidence helps the rankers interpret geometric and global scores.
    geometry = ep.sift_features[:, :, [1, 2, 4]].max(axis=1)
    whole = np.column_stack([ep.query_features, geometry, extra["kernel_q"]])
    broadcast = np.broadcast_to(whole[:, None], (*x.shape[:2], whole.shape[1]))
    candidate_x = np.concatenate([x, broadcast], axis=2).astype(np.float32)
    return cached, candidate_x, whole


def histogram():
    return HistGradientBoostingClassifier(max_iter=115, learning_rate=0.07,
                                          max_leaf_nodes=15, min_samples_leaf=50,
                                          l2_regularization=7, early_stopping=False,
                                          random_state=885)


def light_predict(model, x):
    return model.booster_.predict(x, raw_score=True, num_threads=2)

def snorm(qg, gg, gallery_labels, k=30):
    # Gallery cohorts exclude the reference identity. Query cohorts exclude the
    # provisional winning identity, using only observable gallery labels.
    gg[gallery_labels[:,None]==gallery_labels] = -np.inf
    g_cohort = np.take_along_axis(gg,top(gg,k),axis=1)
    leaders = gallery_labels[qg.argmax(axis=1)]
    cohort = np.where(gallery_labels[None,:]==leaders[:,None],-np.inf,qg)
    q_cohort = np.take_along_axis(cohort,top(cohort,k),axis=1)
    q = (qg-q_cohort.mean(axis=1,keepdims=True))/q_cohort.std(axis=1,keepdims=True).clip(.015)
    g = (qg-g_cohort.mean(axis=1))/g_cohort.std(axis=1).clip(.015)
    return .5*(q+g)

SET = 'amigo_manual_v1'
top_rows = top

@dataclass
class Block:
    episode: object
    labels: np.ndarray
    scores: np.ndarray
    features: np.ndarray
    novelty: np.ndarray
    graph: sparse.csr_matrix


def zscore(x):
    return (x-x.mean(axis=1,keepdims=True))/x.std(axis=1,keepdims=True).clip(1e-6)


def graph_signals(z,group):
    n = len(z)
    ng = len(group)
    nn, vv = [], []
    for offset in range(0,n,128):
        sim = z[offset:offset+128] @ z.T
        sim[np.arange(len(sim)),offset+np.arange(len(sim))] = -2
        top = top_rows(sim,16)
        nn.append(top)
        vv.append(np.take_along_axis(sim,top,axis=1))
    nn,vv = np.vstack(nn),np.vstack(vv)
    strength=np.maximum(vv,0)**4
    graph=sparse.csr_matrix((strength.ravel(),(np.repeat(np.arange(n),16),nn.ravel())),shape=(n,n))
    graph=(graph+graph.T)*.5
    mass=np.asarray(graph.sum(1)).ravel().clip(1e-8)
    scale=sparse.diags(1/np.sqrt(mass))
    laplacian=scale@graph@scale
    y=np.zeros((n,group.max()+1),dtype='f')
    y[np.arange(ng),group]=1/np.sqrt(np.bincount(group)[group])
    current=y.copy()
    for _ in range(7):
        current=y+.77*(laplacian@current)
    class_mass=current[ng:]
    class_mass/=class_mass.sum(1,keepdims=True).clip(1e-8)
    # Correct for the different query/gallery sizes in CV and inference.
    query_frac=((nn[ng:,:10]>=ng).sum(1)+.5)/11
    enrichment=logit(query_frac)-logit((n-ng)/n)
    return np.log(class_mass.clip(1e-7)),enrichment


def class_view_stats(gallery,query,group,vg,vq):
    groups=label_groups(group)
    centroids=[]; columns=[]; tags=[]
    for label,refs in enumerate(groups):
        for view in np.unique(vg[refs]):
            centroids.append(gallery[refs[vg[refs]==view]].mean(0))
            columns.append(label)
            tags.append(view)
    sim=query@unit(np.stack(centroids)).T
    columns=np.asarray(columns)
    best=strongest(sim,columns,len(groups))
    aligned=np.where(vq[:,None]==np.asarray(tags)[None,:],sim,-1)
    return best,strongest(aligned,columns,len(groups))


def pool_peers(raw,z,view):
    sim=raw@raw.T
    np.fill_diagonal(sim,-2)
    nn=top_rows(sim,4)
    ii=np.repeat(np.arange(len(nn)),nn.shape[1]); jj=nn.ravel()
    mutual=np.any(nn[jj]==ii[:,None],axis=1)
    same_side=(view[ii]==view[jj]) | (view[ii]=="") | (view[jj]=="")
    score=np.sum(z[ii]*z[jj],axis=1)
    weights=np.where(mutual & same_side & (sim[ii,jj]>.66) & (score>.60),sim[ii,jj]**6,0)
    return sparse.csr_matrix((weights,(ii,jj)),shape=sim.shape)


def gallery_evidence(data,ep,extras,retrieval_features,geometry,tex,indices,g):
    q=ep.rows[indices]
    species=ep.species[indices[0]]
    g=g[data.species[g]==species]
    g=g[np.argsort(data.label[g],kind='stable')]
    labels,group,counts=np.unique(data.label[g],return_inverse=True,return_counts=True)
    rows=np.r_[g,q]; n=len(g)
    def fit_embedding(power):
        raw=np.sign(data.x[rows])*np.abs(data.x[rows])**.85
        pca=PCA(n_components=320,svd_solver='randomized',random_state=875,iterated_power=3)
        z=pca.fit_transform(unit(raw))
        return unit(z*(pca.explained_variance_+.0015)**(-power/2)).astype('f')
    z=fit_embedding(.58)
    subgroups=label_groups(group)
    g_z,q_z=z[:n],z[n:]
    lda=LinearDiscriminantAnalysis(solver='eigen',shrinkage=.22,
                    n_components=min(160,np.count_nonzero(counts>=2)-1))
    enough=counts[group]>=2
    lda.fit(g_z[enough],group[enough])
    l_z=unit(lda.transform(z)).astype('f')
    wg,wq=fit_wccn(g_z,subgroups,q_z)
    kg,kq=fit_kissme(g_z,subgroups,q_z,reg_s=.14,reg_d=.14)
    cg,cq=fit_cross_view_cca(g_z,subgroups,data.view[g],q_z,data.view[q],
                           max_dim=160,n_comp=72)
    raw=data.x[q]@data.x[g].T
    win=strongest(raw,group,len(labels))
    pca_sim=q_z@g_z.T
    raw_imp = data.x[g]@data.x[g].T
    norm=snorm(raw,raw_imp,group)
    basis=np.concatenate([z,.60*l_z],axis=1)
    Y=np.eye(len(labels),dtype='f')[group]
    gram=basis[:n].T@basis[:n]
    gram.flat[::len(gram)+1]+=4.
    balanced=np.linalg.solve(gram,basis[:n].T@(Y/counts[None,:]**.70))
    ridge=basis[n:]@unit(balanced.T).T
    view,same_view=class_view_stats(g_z,q_z,group,data.view[g],data.view[q])
    diff,enrichment=graph_signals(unit(np.column_stack([.75*z,.55*l_z,.35*np.vstack([kg,kq])])),group)
    f,v=tex
    fv=q_texture(f,g,q,group,len(labels))
    vl=q_texture(v,g,q,group,len(labels))
    matrices=[win,strongest(pca_sim,group,len(labels)),strongest(l_z[n:]@l_z[:n].T,group,len(labels)),
              strongest(wq@wg.T,group,len(labels)),strongest(kq@kg.T,group,len(labels)),
              strongest(cq@cg.T,group,len(labels)),view,same_view,ridge,diff,fv,vl,
              strongest(norm,group,len(labels))]
    # A compact hand-set retrieval fusion with equal input scales. An expanded
    # shortlist also admits identities backed by one specialist.
    weights=[.23,.15,.11,.08,.065,.035,.08,.025,.07,.065,.02,.025,.045]
    fusion=sum(w*zscore(mat) for w,mat in zip(weights,matrices))
    direct=top_rows(fusion,WIDTH)
    specialists=[top_rows(mat,4) for mat in (win,view,ridge,diff,fv,vl)]
    old_index=np.searchsorted(labels,ep.candidates[indices])
    assert np.array_equal(labels[old_index],ep.candidates[indices])
    out=np.empty_like(direct)
    for i in range(len(out)):
        preferred=[*direct[i,:8],*old_index[i,:5]]
        for k in range(3):
            preferred.extend(block[i,k] for block in specialists)
        preferred.extend(direct[i,8:])
        out[i] = list(dict.fromkeys(preferred))[:WIDTH]
    cand=labels[out]
    pick=np.arange(len(q))[:,None]
    old_match=cand[:,:,None]==ep.candidates[indices,None,:]
    old_pos=old_match.argmax(axis=2)
    preserved=retrieval_features[indices][pick,old_pos]
    preserved=np.where(old_match.any(2)[:,:,None],preserved,np.nan)
    extra_geometry=np.zeros((*cand.shape,4),dtype='f')
    matched=old_match.any(2)
    if geometry is not None:
        extra_geometry=np.where(matched[:,:,None],geometry[indices][pick,old_pos],0)
    ordered=np.stack([mat[pick,out] for mat in matrices],axis=2)
    relative=np.stack([mat[pick,out]-mat.max(1,keepdims=True) for mat in matrices],axis=2)
    shape=np.stack([np.log1p(counts[out]),fusion[pick,out],matched],axis=2)
    x=np.concatenate([ordered,relative,shape,extra_geometry,preserved],axis=2).astype('f')
    novelty=[ep.query_features[indices],extras['kernel_q'][indices],
              ep.sift_features[indices][:,:,[1,2,4]].max(1),enrichment]
    # Novelty inputs exclude hard-coded domain priors and query ID counters.
    for mat in [*matrices,fusion]:
        top=np.take_along_axis(mat,top_rows(mat,3),axis=1)
        novelty.append(np.column_stack([top[:,0],top[:,0]-top[:,1],top[:,0]-top[:,2]]))
    novelty=np.column_stack(novelty).astype('f')
    return cand,x,fusion[pick,out],novelty,pool_peers(data.x[q],q_z,data.view[q])


def q_texture(x,g,q,group,n):
    source=x[g].astype('f'); query=x[q].astype('f')
    center=source.mean(0)
    return strongest(unit(query-center)@unit(source-center).T,group,n)


def read_geometry(ep,key):
    path=CACHE/f'top50_v1_{key}.npz'
    if not path.exists():return None
    with np.load(path,allow_pickle=False) as f:
        geom=f['features'][:,:,3:]
    full=(len(ep.rows),ep.candidates.shape[1],4)
    if geom.shape==full:return geom
    positions=training_positions(ep)
    assert len(geom)==len(positions)
    result=np.zeros(full,dtype='f')
    result[positions]=geom
    return result


def make_block(data,folds,key,tex):
    cache=CACHE/f'{SET}_{key}.npz'
    if cache.exists():
        ep,_=validate_inputs(CACHE,key,folds,data)
        with np.load(cache,allow_pickle=False) as f:
            assert np.array_equal(f['rows'],ep.rows)
            graph=sparse.csr_matrix((f['gv'],f['gi'],f['gp']),shape=(len(ep.rows),len(ep.rows)))
            return Block(ep,f['labels'],f['scores'],f['x'],f['novelty'],graph)
    item=retrieval_fold(data,folds,CACHE,key)
    ep,extra=item[0]
    arrays=[]
    geometry=read_geometry(ep,key)
    for sp in SPECIES:
        idx=np.flatnonzero(ep.species==sp)
        arrays.append(gallery_evidence(data,ep,extra,item[1],geometry,tex,idx,folds[key][0]))
        print(key,sp,'features ready',flush=True)
    labels,x,scores,novelty=map(np.concatenate,zip(*(block[:4] for block in arrays)))
    graph=sparse.block_diag([block[-1] for block in arrays],format='csr')
    np.savez_compressed(cache,rows=ep.rows,labels=labels,scores=scores,x=x,novelty=novelty,
                       gv=graph.data,gi=graph.indices,gp=graph.indptr)
    return Block(ep,labels,scores,x,novelty,graph)


def fit_candidates(blocks,exclude_rows=None):
    features=np.concatenate([b.features for b in blocks])
    labels=np.concatenate([b.labels for b in blocks])
    truth=np.concatenate([b.episode.labels for b in blocks])
    correct=labels==truth[:,None]
    use=correct.any(1)
    if exclude_rows is not None:
        queries=np.concatenate([b.episode.rows for b in blocks])
        use &= ~np.isin(queries,exclude_rows)
    x=features[use].reshape(-1,features.shape[2]);y=correct[use].ravel()
    light=light_model(165).fit(x,y.astype(int),group=np.full(use.sum(),WIDTH))
    hist=histogram().fit(x,y,sample_weight=np.where(y,7.,1.))
    return light,hist


def scores(models,block):
    x=block.features.reshape(-1,block.features.shape[2])
    return np.stack([light_predict(models[0],x).reshape(block.scores.shape),
                     models[1].decision_function(x).reshape(block.scores.shape)])


def novelty_inputs(block,margins):
    raw=[]
    for matrix in margins:
        best=np.take_along_axis(matrix,top(matrix,3),axis=1)
        raw.append(np.column_stack([best[:,0],best[:,0]-best[:,1],best[:,0]-best[:,-1]]))
    return np.column_stack([block.novelty,*raw])

def gallery_only_inputs(b):
    # First eleven values are legacy batch statistics. Retain metadata and
    # best gallery scores, omit query density and query-to-query affinity.
    columns = [5,6,7,8,9,29,30,31]
    # Each retrieval method adds (best gallery score, two short margins).
    # Omit diffusion and fused scores, which include query-batch evidence.
    for method in (0,1,2,3,4,5,6,7,8,10,11,12):
        columns.extend(range(33+3*method,36+3*method))
    assert b.novelty.shape[1]==75
    selected = b.novelty[:,columns]
    sift = b.novelty[:,30]
    direct = b.novelty[:,33]
    normalized = b.novelty[:,33+3*12]
    return np.column_stack([selected,sift*direct,sift*np.maximum(normalized,0)])


def fit_raw(parts,targets,profile):
    # Cache only model margins generated from supplied features. Stored labels
    # are never consumed during test scoring or calibration.
    key = hashlib.sha256(b"".join(b.episode.rows.tobytes() for b in [*parts,*targets])).hexdigest()[:20]
    file = CACHE/f"amigo_transfer_raw_{profile}_{key}.npz"
    count = len(parts)
    if file.exists():
        with np.load(file,allow_pickle=False) as stored:
            values = [stored[f"margin{i}"] for i in range(count+len(targets))]
        return values[:count],values[count:]
    held=[]
    for i, block in enumerate(parts):
        trained=fit_candidates([other for j,other in enumerate(parts) if i!=j],
                                  exclude_rows=block.episode.rows)
        held.append(scores(trained,block))
        print(profile,"finished cross-fit",i+1,flush=True)
    trained=fit_candidates(parts)
    predicted=[scores(trained,block) for block in targets]
    temp=file.with_suffix(".partial.npz")
    np.savez_compressed(temp,**{f"margin{i}":x for i,x in enumerate([*held,*predicted])})
    temp.replace(file)
    return held,predicted


def verified_graph(b,data,key,workers=4):
    file=CACHE/f"amigo_transfer_graph_v1_{key}.npz"
    n=len(b.episode.rows)
    if file.exists():
        with np.load(file,allow_pickle=False) as z:
            assert np.array_equal(z['rows'],b.episode.rows)
            return sparse.csr_matrix((z['w'],(z['left'],z['right'])),shape=(n,n))
    graph=b.graph.tocsr()
    graph.eliminate_zeros()
    coo=sparse.triu(graph,1).tocoo()
    rows=b.episode.rows
    pairs=list(zip(rows[coo.row],rows[coo.col]))
    good=[]
    with Verification(data,workers,CACHE) as sift:
        with ThreadPoolExecutor(max_workers=workers) as pool:
            for offset in range(0,len(pairs),64):
                batch=pairs[offset:offset+64]
                sift.stage_images({int(index) for pair in batch for index in pair},pool)
                good.extend(pool.map(sift.compare,batch))
    mask=np.zeros(len(pairs),dtype=bool)
    weights=np.zeros(len(mask))
    if good:
        count,ratio,spread=np.asarray(good).T
        mask=(count>=10)&(ratio>=.55)&(spread>=.20)
        weights=np.minimum(count/30,1)*ratio*np.minimum(spread/.36,1)
    i=coo.row[mask];j=coo.col[mask]; w=weights[mask]
    left=np.r_[i,j];right=np.r_[j,i]; ww=np.r_[w,w]
    np.savez(file,rows=rows,w=ww,left=left,right=right)
    print(key,"verified",int(mask.sum()),"of",len(pairs),"query edges",flush=True)
    return sparse.csr_matrix((ww,(left,right)),shape=(n,n))


def geometry_pool(graph,p):
    # Trust physical matches for movement between ambiguous identities.
    # Suppress contradictory high-confidence neighbours to avoid cascading errors.
    links=graph.tocoo()
    leading=p.argmax(1)
    confidence=p.max(1)
    agree=(leading[links.row]==leading[links.col])
    both_certain=(confidence[links.row]>.80)&(confidence[links.col]>.80)
    accept=agree | ~both_certain
    weights=links.data[accept]
    matrix=sparse.csr_matrix((weights,(links.row[accept],links.col[accept])),shape=graph.shape)
    total=np.asarray(matrix.sum(1)).ravel()
    signal=(matrix@p)/total[:,None].clip(1e-12)
    alpha=.40*np.clip(total/.55,0,1)[:,None]
    return (1-alpha)*p+alpha*signal

def log_odds(model, features):
    return logit(np.clip(model.predict_proba(features)[:, 1], 1e-5, 1 - 1e-5))


def temperature(scores, classes, labels):
    match = classes == labels[:, None]
    keep = match.any(axis=1)
    s, correct = scores[keep], match[keep].argmax(axis=1)
    scale = float(np.median(np.std(s, axis=1)))
    search = np.maximum(scale, 1e-5) * np.geomspace(0.06, 1.30, 25)
    rows = np.arange(len(correct))
    nll = [-np.log(np.maximum(softmax(s / t, axis=1)[rows, correct], 1e-12)).mean()
           for t in search]
    return float(search[np.argmin(nll)])


def estimate_prior(odds, center=0.30, shrink=0.65):
    # The novelty models were fitted with a 50:50 prior within each species.
    prior = center
    for _ in range(30):
        update = float(expit(odds + logit(prior)).mean())
        if abs(update - prior) < 0.0001:
            break
        prior = float(np.clip(update, 0.08, 0.70))
    return float(np.clip(center + shrink * (prior - center), 0.12, 0.62))

def side_group(angles):
    return np.where((angles >= 5) & (angles <= 7), -1,
                    np.where((angles >= 1) & (angles <= 3), 1, 0))

@dataclass
class CameraEpisode:
    block: object
    camera: np.ndarray
    context: np.ndarray


def camera_evidence(data, gallery, query, candidates):
    """Candidate-side catalogue coverage and side-conditioned match evidence."""
    gallery = gallery[np.argsort(data.label[gallery], kind="stable")]
    labels, group, counts = np.unique(data.label[gallery], return_inverse=True, return_counts=True)
    index = np.searchsorted(labels, candidates)
    assert np.array_equal(labels[index], candidates)
    gal_side = side_group(data.angle[gallery])
    query_side = side_group(data.angle[query])
    g = data.x[gallery]
    exact_views = np.column_stack([
        np.bincount(group, weights=(data.view[gallery] == angle), minlength=len(labels))
        for angle in np.unique(data.view[query])
    ])
    exact = exact_views[index, np.searchsorted(np.unique(data.view[query]), data.view[query])[:, None]]
    features = np.zeros((*candidates.shape, 11), dtype=np.float32)
    features[:, :, 0] = np.log1p(exact)
    for side_value in (-1, 0, 1):
        key = np.flatnonzero(query_side == side_value)
        if not len(key):
            continue
        same = (gal_side == side_value) & (side_value != 0)
        opposite = (gal_side == -side_value) & (side_value != 0)
        same_n = np.bincount(group, weights=same, minlength=len(labels))
        opposite_n = np.bincount(group, weights=opposite, minlength=len(labels))
        chosen = index[key]
        features[key, :, 1:5] = np.stack([
            np.log1p(same_n[chosen]), np.log1p(opposite_n[chosen]),
            (same_n[chosen] > 0) & (opposite_n[chosen] > 0),
            (same_n[chosen] / counts[chosen]),
        ], axis=2)
        for mask, start in [(same, 5), (opposite, 8)]:
            if not mask.any():
                continue
            for batch in np.array_split(key, (len(key) + 191) // 192):
                sim = data.x[query[batch]] @ g.T
                sim[:, ~mask] = -2
                best = strongest(sim, group, len(labels))
                hits = best[np.arange(len(batch))[:, None], index[batch]]
                first_two = np.take_along_axis(best, top(best, 2), axis=1)
                rival = np.where(index[batch] == best.argmax(1)[:, None], first_two[:, 1, None],
                                 first_two[:, 0, None])
                valid = (hits > -1.5) & (rival > -1.5)
                features[batch, :, start] = np.where(hits > -1.5, hits, 0)
                features[batch, :, start + 1] = np.where(valid, hits - rival, 0)
                features[batch, :, start + 2] = valid
    return features


def augment(data, gallery, block):
    ep = block.episode
    camera = np.zeros((*block.labels.shape, 11), dtype=np.float32)
    for sp in SPECIES:
        rows = np.flatnonzero(ep.species == sp)
        camera[rows] = camera_evidence(data, gallery[data.species[gallery] == sp],
                                      ep.rows[rows], block.labels[rows])
    angle = data.angle[ep.rows]
    context = np.column_stack([ep.species == "hyena", angle < 0, angle == 0, angle == 4,
                               side_group(angle) == -1, side_group(angle) == 1])
    return CameraEpisode(block, camera, context)


def joint_features(item):
    block, camera = item.block, item.camera
    n, width = block.labels.shape
    novelty = np.column_stack([block.novelty, item.context, camera[:, :, 5:].max(axis=1)])
    feature = np.full((n, width + 1, block.features.shape[2] + camera.shape[2] +
                       novelty.shape[1] + 1), np.nan, dtype=np.float32)
    feature[:, :-1, :block.features.shape[2]] = block.features
    feature[:, :-1, block.features.shape[2]:-novelty.shape[1]-1] = camera
    feature[:, :, -novelty.shape[1]-1:-1] = novelty[:, None, :]
    feature[:, :, -1] = (np.arange(width + 1) == width).astype(np.float32)
    return feature


def fit_joint(parts, predict, excluded=None):
    x = np.concatenate([joint_features(item) for item in parts])
    labels = np.concatenate([item.block.labels for item in parts])
    truth = np.concatenate([item.block.episode.labels for item in parts])
    rows = np.concatenate([item.block.episode.rows for item in parts])
    species = np.concatenate([item.block.episode.species for item in parts])
    matches = labels == truth[:, None]
    novel = truth == NEW
    use = matches.any(1) | novel
    if excluded is not None:
        use &= ~np.isin(rows, excluded)
    x, y = x[use], np.column_stack([matches, novel])[use]
    width = x.shape[1]
    flat, target = x.reshape(-1, x.shape[2]), y.ravel().astype(int)
    weights = np.ones(len(x))
    # Equal known/novel weights within species reduce sensitivity to the split's
    # chosen mix. The supervised novelty stage estimates transfer separately.
    for sp in SPECIES:
        part = species[use] == sp
        for flag in (True, False):
            group = part & (novel[use] == flag)
            weights[group] = .5 * part.sum() / max(group.sum(), 1)
    cat = CatBoostRanker(iterations=145, depth=5, learning_rate=.064, l2_leaf_reg=13,
                         random_seed=2026, thread_count=2, loss_function="QuerySoftMax",
                         verbose=False, allow_writing_files=False)
    cat.fit(flat, target, group_id=np.repeat(np.arange(len(y)), width),
            group_weight=np.repeat(weights, width))
    xgb = XGBRanker(n_estimators=135, max_depth=4, learning_rate=.06, reg_lambda=15,
                    min_child_weight=14, subsample=.89, colsample_bytree=.82,
                    objective="rank:ndcg", lambdarank_num_pair_per_sample=7,
                    tree_method="hist", n_jobs=2, random_state=2026)
    xgb.fit(flat, target, group=np.full(len(y), width), sample_weight=weights)
    estimates = []
    for item in predict:
        x = joint_features(item)
        z = x.reshape(-1, x.shape[2])
        estimates.append(np.stack([model.predict(z).reshape(x.shape[:2]) for model in (cat, xgb)]))
    return estimates


def joint_crossfits(parts, targets, key):
    fingerprint = hashlib.sha256(b"".join(item.block.episode.rows.tobytes()
                                          for item in [*parts, *targets])).hexdigest()[:16]
    path = CACHE / f"amigo_portfolio_joint_v1_{key}_{fingerprint}.npz"
    if not path.exists():
        values = []
        for index, item in enumerate(parts):
            others = [part for j, part in enumerate(parts) if j != index]
            values.extend(fit_joint(others, [item], item.block.episode.rows))
            print(key, "joint ranker cross-fit", index + 1, flush=True)
        values.extend(fit_joint(parts, targets))
        temp = path.with_suffix(".partial.npz")
        np.savez_compressed(temp, *values)
        temp.replace(path)
    with np.load(path, allow_pickle=False) as stored:
        values = [stored[f"arr_{i}"] for i in range(len(parts) + len(targets))]
    return values[:len(parts)], values[len(parts):]


def uncalibrated_features(item, base, joint, stable):
    b = item.block
    # Describe the two leading identities. Only the observed candidates' side
    # coverage is used. Gallery coverage for the true identity is audit-only.
    blend = .5 * base[0] / base[0].std(axis=1, keepdims=True).clip(.01) + \
            .5 * base[1] / base[1].std(axis=1, keepdims=True).clip(.01)
    leaders = top(blend, 2)
    position = np.arange(len(leaders))[:, None]
    camera = item.camera[position, leaders]
    key_signals = b.features[position, leaders][:, :, [0, 1, 2, 6, 7, 8, 10, 11, 12, 26, 29, 30, 31, 32]]
    lead = np.column_stack([camera[:, 0], key_signals[:, 0], camera[:, 1, [5, 6, 8, 9]],
                            key_signals[:, 0] - key_signals[:, 1], item.context])
    inputs = gallery_only_inputs(b) if stable else novelty_inputs(b, base)
    if stable:
        return np.column_stack([inputs, lead])
    extra = [score[:, -1] - score[:, :-1].max(1) for score in joint]
    return np.column_stack([inputs, lead, *extra])


def fit_calibrators(parts, base_margins, joint_margins):
    y = np.concatenate([item.block.episode.novel for item in parts])
    species = np.concatenate([item.block.episode.species for item in parts])
    full = np.concatenate([uncalibrated_features(*items, False)
                           for items in zip(parts, base_margins, joint_margins)])
    stable = np.concatenate([uncalibrated_features(*items, True)
                             for items in zip(parts, base_margins, joint_margins)])
    # Compress sample weights for queries appearing in multiple inner episodes.
    rows = np.concatenate([item.block.episode.rows for item in parts])
    _, group, count = np.unique(rows, return_inverse=True, return_counts=True)
    weight = 1 / count[group]
    for sp in SPECIES:
        for label in (True, False):
            use = (species == sp) & (y == label)
            weight[use] *= .5 * (species == sp).sum() / weight[use].sum()
    logistic = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                             LogisticRegression(C=.035, max_iter=450))
    logistic.fit(stable, y, logisticregression__sample_weight=weight)
    trees = make_pipeline(SimpleImputer(strategy="median"),
                         ExtraTreesClassifier(n_estimators=210, max_features=.68,
                                              min_samples_leaf=22, max_depth=11, n_jobs=3,
                                              random_state=26120))
    trees.fit(full, y, extratreesclassifier__sample_weight=weight)
    return logistic, trees


def joint_temperatures(raw, parts):
    truth = np.concatenate([item.block.episode.labels for item in parts])
    labels = np.concatenate([item.block.labels for item in parts])
    labels = np.column_stack([labels, np.full(len(labels), NEW)])
    return [temperature(np.concatenate([score[j] for score in raw]), labels, truth)
            for j in range(len(raw[0]))]


def dense(labels, probabilities, vocab):
    result = np.zeros((len(labels), len(vocab) + 1))
    result[np.arange(len(labels))[:, None], np.searchsorted(vocab, labels)] = probabilities[:, :-1]
    result[:, -1] = probabilities[:, -1]
    return result


def fit_graph_calibration(parts, held):
    truth = np.concatenate([block.episode.labels for block in parts])
    species = np.concatenate([block.episode.species for block in parts])
    features = np.concatenate([novelty_inputs(block, scores) for block, scores in zip(parts, held)])
    unseen = truth == NEW
    weights = np.ones(len(unseen))
    for sp in SPECIES:
        group = species == sp
        for label in (False, True):
            use = group & (unseen == label)
            weights[use] = 0.5 * group.sum() / max(use.sum(), 1)
    logistic = make_pipeline(StandardScaler(), LogisticRegression(C=.05,max_iter=500))
    logistic.fit(features, unseen, logisticregression__sample_weight=weights)
    hist = HistGradientBoostingClassifier(max_iter=105, learning_rate=.085, max_leaf_nodes=15,
                                         min_samples_leaf=55, l2_regularization=6.0,
                                         early_stopping=False, random_state=392)
    hist.fit(features, unseen, sample_weight=weights)
    labels = np.concatenate([block.labels for block in parts])
    temps = [temperature(np.concatenate([m[k] for m in held]), labels, truth) for k in (0,1)]
    return logistic, hist, temps


def portfolio_posterior(data, item, base, joint, graph_fit, side_models, joint_temps, vocab):
    block = item.block
    ep = block.episode
    logistic, hist, temperatures = graph_fit
    known = .76*softmax(base[0]/temperatures[0],axis=1) + .24*softmax(base[1]/temperatures[1],axis=1)
    graph = verified_graph(block, data, "test", workers=WORKERS)
    features = novelty_inputs(block,base)
    odds = .50*log_odds(logistic,features) + .50*log_odds(hist,features)
    graph_p = geometry_pool(graph, dense_posterior((block.labels,known,odds),ep.species,vocab))
    side_logistic, side_forest = side_models
    forest = log_odds(side_forest, uncalibrated_features(item,base,joint,False))
    linear = log_odds(side_logistic, uncalibrated_features(item,base,joint,True))
    side_p = geometry_pool(graph, dense_posterior(
        (block.labels,known,.55*forest+.45*linear),ep.species,vocab))
    cat,xgb = [dense(block.labels, softmax(margin/t,axis=1),vocab)
               for margin,t in zip(joint,joint_temps)]
    joint_p = .50*cat + .50*xgb
    hyena = (ep.species=="hyena")[:,None]
    weights = [np.where(hyena, h, l) for h,l in ((.50,.15),(.40,.70),(.10,.15))]
    return sum(weight*p for weight,p in zip(weights,(graph_p,side_p,joint_p)))


In [ ]:
%%model_section leopard_fusion
'Fuse Leopard identities and write the single requested CSV.'
from pathlib import Path
from types import SimpleNamespace
import hashlib

import numpy as np
anchor = notebook_sections['geometry_verification']
spatial = notebook_sections['spatial_ranker']
scored = notebook_sections['species_portfolio']


def build_or_load_gallery(data, folds, names, rebuild):
    cache = scored.CACHE
    required = [cache/f"{key}.npz" for key in names]
    if any(not path.exists() for path in required):
        if not rebuild:
            raise FileNotFoundError("Set CACHE_ROOT to the completed gallery feature cache")
        store = scored.SiftStore(data, scored.WORKERS)
        try:
            scored.prepare(data, folds, store, cache, names)
        finally:
            store.close()
    manual = [cache/f"{scored.SET}_{key}.npz" for key in names]
    textures = None
    if any(not path.exists() for path in manual):
        if not rebuild:
            raise FileNotFoundError("The 24-candidate feature cache is required")
        with scored.Verification(data, scored.WORKERS, cache) as verify:
            for key in names:
                scored.prepare_geometry(data, folds, key, verify)
            textures = scored.texture_codes(data, folds, names, verify)
    return [scored.make_block(data, folds, key, textures) for key in names]


def leopard_spatial(data,folds,names,rebuild):
    cache = anchor.CACHE/"animal_v3_v1"/"known_fusion_v1"/"test.npz"
    if cache.exists():
        with np.load(cache,allow_pickle=False) as saved:
            order = {image:i for i,image in enumerate(saved["ids"])}
            rows = folds["test"][1][data.species[folds["test"][1]]=="leopard"]
            return rows,saved["vocabulary"],saved["animal_v3"][[order[image] for image in data.id[rows]]]
    completed = [spatial.f.CACHE/f"{key}_leopard_all.npz" for key in [*names,"test"]]
    if any(not path.exists() for path in completed):
        if not rebuild:
            raise FileNotFoundError("The Leopard spatial-model feature cache is required")
        spatial.f.build([*names,"test"],scored.WORKERS,data=data,folds=folds)
    parts = [spatial.load(data,folds,key,"leopard") for key in names]
    target = spatial.load(data,folds,"test","leopard")
    held, raw = spatial.margins(parts,[target],"test","leopard")
    recipe = SPATIAL_RECIPE
    vocab, old = spatial.aligned_ref(data,target,"test")
    novelty = spatial.fit_novelty(parts,held)
    p = spatial.track_a(target,raw[0],recipe["temperature"],recipe["rank_weights"],
                        spatial.novelty_log_odds(novelty,target,raw[0]),vocab)
    q = spatial.track_b(p,target.graph)
    return target.rows,vocab,sum(w*x for w,x in zip(recipe["blend"],(p,q,old)))

def generate_leopard_submission(data_root, cache_root, output, rebuild=False):
    anchor.ROOT, anchor.CACHE = Path(data_root), Path(cache_root)
    scored.CACHE = anchor.CACHE
    spatial.f.CACHE = anchor.CACHE / "animal_v3_v1"
    anchor.CACHE.mkdir(parents=True, exist_ok=True)
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    data = scored.Data(anchor.ROOT, need_sift=rebuild)
    names = [f"full_inner_{i}" for i in range(3)]
    folds = dict(zip(names, scored.calibration_splits(data, data.train_rows)))
    folds["test"] = data.train_rows, data.test_rows
    blocks = build_or_load_gallery(data, folds, [*names, "test"], rebuild)
    train, target = blocks[:-1], blocks[-1]
    episode = target.episode
    if not rebuild and not (anchor.CACHE / "amigo_transfer_graph_v1_test.npz").is_file():
        raise FileNotFoundError("Set CACHE_ROOT to the saved SIFT graph and feature caches")
    parts = [scored.augment(data, folds[key][0], block) for key, block in zip(names, train)]
    item = scored.augment(data, folds["test"][0], target)
    held, base = scored.fit_raw(train, [target], "test")
    joint_held, joint = scored.joint_crossfits(parts, [item], "test")
    vocab = np.union1d(scored.possible_labels(scored.validate_inputs(anchor.CACHE,"test",folds,data)),
                       target.labels)
    posterior = scored.portfolio_posterior(data, item, base[0], joint[0],
        scored.fit_graph_calibration(train, held), scored.fit_calibrators(parts, held, joint_held),
        scored.joint_temperatures(joint_held, parts), vocab)

    spatial_cache = spatial.f.CACHE / "known_fusion_v1" / "test.npz"
    anchor_cache = anchor.CACHE / "cross_reference_test.npz"
    if not spatial_cache.exists() and not anchor_cache.exists():
        if not rebuild:
            raise FileNotFoundError("A cached spatial posterior or its reference is required")
        anchor.cross.current_probabilities(data, folds, anchor.CACHE, names, ["test"])
    spatial_rows, spatial_vocab, spatial_prob = leopard_spatial(data, folds, names, rebuild)
    spatial_index = {row: i for i, row in enumerate(spatial_rows)}
    predictions = scored.ranked(posterior, vocab)
    leopard = episode.species == "leopard"
    hyena = episode.species == "hyena"
    hyena_predictions = predictions[hyena].copy()
    gallery = data.train_rows[data.species[data.train_rows] == "leopard"]
    identities, counts = np.unique(data.label[gallery], return_counts=True)
    def extract(p, labels):
        indices = np.searchsorted(labels, identities)
        np.testing.assert_array_equal(labels[indices], identities)
        return p[:, np.r_[indices, len(labels)]]
    old = extract(posterior[leopard], vocab)
    new = extract(spatial_prob[[spatial_index[row] for row in episode.rows[leopard]]], spatial_vocab)
    def conditional(p):
        known = p[:, :-1]
        return known / known.sum(1, keepdims=True)
    weight = .45 * np.where(counts <= 2, .25, 1.)
    known = (1 - weight) * conditional(old) + weight * conditional(new)
    known /= known.sum(1, keepdims=True)
    p_new = old[:, -1]
    final = np.column_stack([known * (1 - p_new[:, None]), p_new])
    predictions[leopard] = scored.ranked(final, identities)
    np.testing.assert_array_equal(predictions[hyena], hyena_predictions)
    assert predictions.shape == (2791, 5) and hyena.sum() == 1217
    counts = [int(np.count_nonzero((episode.species==sp)&(predictions[:,0]==scored.NEW)))
              for sp in scored.SPECIES]
    path = output / "submission_portfolio_leopard_v3.csv"
    scored.write_submission(data, SimpleNamespace(rows=episode.rows), predictions, path)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{len(predictions):,} predictions. Rank-one unseen counts {dict(zip(scored.SPECIES, counts))}")
    print(f"SHA-256 {digest}. Exact match to the saved CSV {digest == EXPECTED_SHA256}")
    return path


SPATIAL_RECIPE = {'blend': [0.0, 0.8, 0.2],
 'inner_prior_scores': [[0.8939533013844515,
                         0.900436014533609,
                         0.9069187276827665,
                         0.913401440831924],
                        [0.7321828760692151,
                         0.7711547964274745,
                         0.8101267167857341,
                         0.8490986371439935],
                        [0.8988723713924628,
                         0.9133182194874344,
                         0.9277640675824064,
                         0.942209915677378]],
 'objective': 0.8297886927790307,
 'rank_weights': [0.33436718094765816,
                  0.13936029363990343,
                  0.13844898753541623,
                  0.23953555781066915,
                  0.14828798006635302],
 'temperature': [0.8220979465563597,
                 0.7257928725954874,
                 0.9508284631739101,
                 0.8968204226887992,
                 1.1994185437703828,
                 3.844995777065022]}
EXPECTED_SHA256 = '21d24bb4faf2b84a798d384311e360e68f45e867e23942ef830fd0ef9755e5f2'


## Generate the submission

In [ ]:
notebook_sections["leopard_fusion"].generate_leopard_submission(
    DATA_ROOT, CACHE_ROOT, OUTPUT, REBUILD_FEATURES
)
